# 가지 YOLO — Kaggle 학습 전용 (Save & Run All) — v4.24

브라우저를 닫아도 학습이 계속되게 하는 노트북입니다. 셀은 본 노트북의 **0번 셀**과 **15번 셀**을 그대로 복사한 것입니다.

1. 아래 15번 셀 맨 위 설정(`EXP_DIR`, `EXP_QUERY`, `TRAIN_NAME` …)과 0번 셀 `OPERATOR`·`REVIEWED_RULE`·`VAL_RULE`을 정합니다. 두 사람이 같은 때 학습하면 `TRAIN_NAME`을 다르게.
2. 먼저 본 노트북에서 15번을 `LIST_ONLY = True`로 실행해 train/val 비율 표를 확인하세요.
3. 오른쪽 Session options: Accelerator **GPU**, Internet **On**, Add-ons → Secrets에서 `RCLONE_CONF` 체크.
4. 오른쪽 위 **Save Version → Save & Run All (Commit)**. 최대 12시간, 주간 GPU 한도 안에서 실행됩니다.
5. 결과는 에폭마다 Drive `runs/<이름>/`에, TASK별 train/val은 `logs/train_tasks.csv`에 올라갑니다. 끝나면 본 노트북 0번 셀 `MODEL_CHOICE`.
6. 중간에 끊기면 같은 설정으로 다시 Save & Run All → 마지막 에폭부터 이어서 학습합니다.
7. 웹 화면의 학습 요청으로 돌리려면 15번 셀 `FROM_REQUESTS = True` (요청의 export 폴더·검색어·학습 이름을 씀, 끝나면 요청이 '완료').


In [ ]:
# 0. 공통 설정 · 준비 — 세션이 새로 열릴 때마다 가장 먼저 실행 (다시 실행해도 안전) · 앱 버전 4.30.1 (Colab · Kaggle)
APP_VERSION = '4.31.0'             # 이 노트북의 버전 — Drive scripts/의 파일도 같은 버전이어야 함 (바꾸지 마세요)
# ====================== 설정 (여기만 고치세요) ======================
BASE     = '/content/drive/MyDrive/eggplant_yolo'   # 공유받은 폴더는 '내 드라이브'에 바로가기를 만들면 이 경로로 보임
CREATE_BASE = False            # True = BASE가 없을 때 새로 만듦 (처음 설치하는 사람만)
MODEL_CHOICE = 'latest'        # 쓸 학습 모델: 'latest' = runs/에서 가장 최근에 학습을 끝낸 모델 (15번 셀 결과가 자동으로 잡힘)
                               #   또는 아래 목록 번호(1, 2 …) · 또는 runs/ 폴더 이름 그대로 (예: 'bud_r1')
MODEL_NAMES = {                # 자주 쓰는 모델에 번호를 붙여 둠 (선택 사항)
    1: 'fruit_1920',                                   # 예전 dataset.zip으로 학습
    2: 'exp_20261025_081411_cvat_for_images_1_1',      # exports/20261025_081411_cvat_for_images_1_1로 학습 (15번 셀)
}
MODEL    = 'yolo11s.pt'        # 사전학습 가중치 (s → 더 높이려면 m)
IMGSZ    = 1920                # 변환 때 TRAIN_IMGSZ와 같게
EPOCHS   = 100
PATIENCE = 30                  # 개선이 없으면 일찍 끝냄
BATCH    = -1                  # -1 = GPU 메모리에 맞춰 자동. 메모리 오류 시 2 또는 1
DEVICE   = None                # None = 자동(GPU 있으면 GPU)
TRACK_CONF = 0.25             # track 생성 시 탐지 신뢰도 하한. 검출이 적으면 0.1~0.15로 (추적기 기준도 함께 낮아짐)
TRACK_FILL_GAPS = 3            # 잠깐(태스크 프레임 3개 이하) 안 보인 열매는 track을 끊지 않고 CVAT 보간에 맡김
TRACK_BUFFER = 90              # 추적기가 가려진 열매의 ID를 기억하는 프레임 수 (30fps 기준 3초)
TRACK_DECODER = 'opencv'       # 영상 읽기: 'opencv' / 'pyav'(CVAT와 같은 방식) — 20번 셀 진단 결과에 따라
TRACK_FRAME_OFFSET = 0         # 영상 프레임 + 이 값 = CVAT 프레임 (20번 셀 진단이 알려 줌)
TRACK_BOX_SOURCE = 'detection'  # 'detection' = 모델이 그 프레임에서 찾은 박스 (권장) / 'tracker' = 추적기 추정 박스(이전 방식, 늦게 따라와 잘림)
TRACK_NEAR = 1                 # 태스크 프레임에 검출이 없을 때 앞뒤 몇 프레임까지 빌려올지 (크면 카메라 이동만큼 밀림)
UPLOAD_MIN_PRECISION = 0.8     # 업로드 권장 기준 (검증 2단계 정밀도) — 대장에 '권장/재검토'로 표시
AA_CONF    = 0.25              # 프레임별 자동 라벨링 신뢰도 하한
REVIEWED_RULE = 'vf'          # 검토 완료 TASK = 이름의 상태에 이 글자가 있음 (vf, vf_r2, vf_r2df_ok, …_vf_AA). '' = 모두 검토 완료로 봄
                              #   검색어에서는 '검토'로 씀: '검토 date=261019..261022' (검색어 'vf'는 상태가 정확히 vf인 것만 — 다름)
VAL_RULE   = 'plant=3|plant=7'      # val(평가) TASK = 이 검색어에 맞는 TASK — 모든 온실·라인의 3번·7번 그루. 15번이 TASK 단위로 train/val을 나눔
                              #   '' = 예전처럼 무작위 영상(CONVERT의 VAL_RATIO) · 예: 'gh=G4', 'row=2 plant=6..10' (웹 학습 요청이 바꿀 수 있음)
CVAT_PROFILES = {             # 계정(프로필) 목록. 이름은 자유 — 셀 맨 위 LOGIN에 이 이름을 씀
    'token':    {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN'},                    # PAT (Google 로그인 계정)
    'password': {'host': 'https://app.cvat.ai', 'login': 'password', 'user': 'CVAT_USER', 'pass': 'CVAT_PASS'}, # 공동 아이디 (ID/PASS)
    'B':        {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN_B'},                  # B 계정 PAT (보안 비밀 CVAT_TOKEN_B)
}                              # 'token' / 'user' / 'pass' 값 = Colab 보안 비밀 이름 (계정마다 다른 이름으로)
CVAT_SDK_VERSION = 'auto'      # 'auto' = 처음 접속할 서버와 같은 X.Y 버전의 cvat-cli/cvat-sdk 설치 (SaaS 자동 업그레이드 대응)
                               # 문제가 생기면 확인된 버전으로 고정: 예 '2.79.0'
CVAT_AUTH  = 'auto'            # 처음 접속할 프로필: 'auto' = 보안 비밀이 있는 첫 프로필 · 또는 프로필 이름
                               # 셀마다 맨 위 LOGIN = '프로필 이름'으로 계정·서버를 바꿀 수 있음
CVAT_ORG     = 'horti'        # 기본 조직: CVAT 조직의 slug 또는 이름 (CVAT 화면 왼쪽 위 조직 메뉴에 보이는 것)
                               #   '' = 개인 작업공간 · None = 구분 안 함 (보이는 모든 조직의 TASK — v4.18까지와 같음)
CVAT_PROJECT = 'eggplant'     # 기본 프로젝트: 이름 또는 ID (그 조직 안). '' = 모든 프로젝트 · '-' = 프로젝트 없는 TASK만
                               #   정하면 셀의 TASK 목록·검색·백업·export가 이 범위만 보고, 16·17번 셀의 새 TASK도 이 조직·프로젝트에 만듦
                               #   TASK ID를 직접 적으면 프로젝트는 안 따짐(조직은 따짐). 프로필마다 다르게: CVAT_PROFILES 항목에 'org': …, 'project': …
                               #   웹 관리 화면의 요청은 그 TASK들의 조직으로 실행 (요청이 끝나면 다시 이 기본값)
DS_ROOT    = '/content/ds'     # 데이터셋을 풀 곳 (Colab 로컬 디스크 — 빠름)
OPERATOR   = ''               # 학습 잠금·완료 기록에 남길 내 이름 (예: 'A-재웅', 'B-james'). 사람마다 다르게
LOCK_STALE_MIN = 45           # 학습 잠금이 이 시간(분) 넘게 갱신되지 않으면 끊긴 학습으로 보고 넘겨받을 수 있음
LOCK_CONFIRM_SEC = 30         # 잠금을 잡은 뒤 이만큼 기다렸다 다시 확인 (두 컴퓨터가 거의 동시에 시작한 경우를 잡음)
# ---- Kaggle에서 실행할 때만 (v4.12) — Colab에서는 무시됨. 준비 방법은 KAGGLE.md ----
KAGGLE_REMOTE   = 'gdrive:'              # rclone 원격. root_folder_id = eggplant_yolo 폴더 ID로 만들었으면 'gdrive:' · 아니면 'gdrive:eggplant_yolo'
KAGGLE_BASE     = '/tmp/eggplant_yolo'   # Kaggle 안의 작업 폴더 (세션이 끝나면 사라짐 — 원본은 언제나 Drive)
KAGGLE_PULL     = ['exports']            # 0번 셀이 처음에 받아 둘 폴더. 영상·백업·TASK 폴더는 셀이 필요할 때 자동으로 받음
KAGGLE_PULL_DATASET = False              # True = dataset.zip · datasets/<선택 모델>.zip도 받음 (2·3번 셀용, 클 수 있음)
KAGGLE_VIDEOS   = ''                     # 미리 받아 둘 영상 (TASK 이름 검색 규칙, 예 'G6 date=261019'). '' = 필요할 때만
KAGGLE_AUTO_SYNC = True                  # 셀마다: 실행 전 공유 기록 받기 → 실행 후 바뀐 파일을 Drive에 올리기 (끄면 drive_sync_now()로 직접)
# ====================================================================
import os, sys, glob, re, json, zipfile, shutil, subprocess, importlib, importlib.util
from pathlib import Path
from urllib.parse import urlparse
import time

# ---------- Kaggle에서 실행 (v4.12) — Drive를 붙일 수 없으므로 rclone으로 받고 올림. Colab에서는 아무것도 안 함 ----------
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
KAGGLE_BATCH = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'      # Save & Run All(커밋)로 실행 중
LOCAL_TMP = '/tmp' if ON_KAGGLE else '/content'                         # 잠깐 쓰는 로컬 폴더
_KG = globals().get('_KG') or {}                                        # 이 세션의 동기화 상태 (0번 셀을 다시 실행해도 유지)
for _k, _v in {'ls': {}, 'media': set(), 'tasks': set(), 'lock_t': {}, 'bg': None, 'secrets': {}}.items():
    _KG.setdefault(_k, _v)
_KG_SHARED = ['logs/**', 'task_registry.csv', 'backup_index.csv', 'backup_status.csv',
              'backups/pending_backups.csv', 'exports/*.csv', 'requests/*.json']   # 셀마다 실행 전에 받는 공유 기록·웹 요청 (작음)
_KG_NOSYNC = ('raw_images/', 'cvat_frames/')                            # 프레임 캐시: 올리지도 받지도 않음 (다시 만들 수 있음)
_KG_NODELETE = ('videos/', 'backups/')                                  # Kaggle에서 지워도 Drive에서는 지우지 않음

def _kg_remote(rel=''):
    r = KAGGLE_REMOTE if KAGGLE_REMOTE.endswith(':') else KAGGLE_REMOTE.rstrip('/') + '/'
    return r + rel.strip('/')

_KG_RC_LOCK = globals().get('_KG_RC_LOCK') or __import__('threading').RLock()

def _kg_rc(args, quiet=False):
    # rclone 실행. 실패하면 마지막 오류만 보여 줌 (값·토큰은 출력에 나오지 않음)
    # v4.28.1: Drive에 쓰는 rclone은 한 번에 하나 — 작업자 셀의 심장 박동 스레드와 본 작업이 동시에 새 폴더를 만들면
    #          Google Drive에 같은 이름 폴더가 둘 생겨(relabel/ 등) 앱과 노트북이 서로 다른 쪽을 봄
    with _KG_RC_LOCK:
        r = subprocess.run(['rclone', *args], capture_output=True, text=True)
    if r.returncode and not quiet:
        err = '\n'.join(l for l in r.stderr.strip().splitlines() if 'ERROR' in l or 'Failed' in l or 'error' in l)[-600:]
        print(f'⚠️ rclone {args[0]} 실패 (코드 {r.returncode}): {err or r.stderr.strip()[-300:]}')
    return r

_KG_SHARDED = ('task_registry.csv', 'backup_index.csv', 'backups/pending_backups.csv', 'exports/pending_exports.csv',
               'logs/uploads.csv', 'logs/activity.csv', 'logs/train_tasks.csv', 'logs/model_compare.csv')

def _kg_filters(paths):
    # 'runs/a' → 파일 또는 폴더 전체 · '*'가 있으면 rclone 규칙 그대로 ('*'는 / 를 넘지 않음, '**'는 넘음)
    out = []
    for p in paths:
        p = '/' + p.strip('/')
        out += ['--include', p] if '*' in p else ['--include', p, '--include', p + '/**']
    if any(str(p).strip('/') in _KG_SHARDED for p in paths):     # v4.31.0: 그 기록의 세션 파일도 함께 (logs/shards/)
        out += ['--include', '/logs/shards/**']
    return out

def drive_pull(*paths, quiet=False):
    # Drive → Kaggle (BASE 기준 경로·패턴). 로컬이 더 새것이면 건너뜀. Colab에서는 아무것도 안 함
    if not ON_KAGGLE or not paths:
        return
    flt = _kg_filters(paths) + [x for n in _KG_NOSYNC for x in ('--exclude', f'/{n}**')]
    r = _kg_rc(['copy', _kg_remote(), BASE, '--update', '--transfers', '8', '--checkers', '16', '-v', '--stats', '0', *flt], quiet=quiet)
    for _l in (r.stderr or '').splitlines():       # 셀 안에서 Drive에서 받은 파일 → 셀이 끝날 때 다시 올리지 않음 (v4.24.2)
        _m = re.search(r'INFO\s+:\s+(.+?): Copied', _l)
        if _m:
            try:                                   # 받은 그대로인지 알려고 크기·수정 시각을 기억 (셀이 그 뒤에 고치면 올림)
                _st = os.stat(os.path.join(BASE, _m.group(1).strip()))
                _KG.setdefault('pulled', {})[_m.group(1).strip()] = (_st.st_mtime_ns, _st.st_size)
            except OSError:
                pass

def kg_sync_exports():
    # Kaggle: Drive exports/에서 새로 생긴 zip을 받아 둠 — 0번 셀 뒤에 다른 세션·컴퓨터가 받은 export도 보이게 (v4.23)
    if ON_KAGGLE:
        drive_pull('exports', quiet=True)

def drive_push(*paths, quiet=False):
    # Kaggle → Drive (BASE 기준 경로·패턴). Drive가 더 새것이면 건너뜀. 지우지는 않음
    if not ON_KAGGLE or not paths:
        return
    _kg_rc(['copy', BASE, _kg_remote(), '--update', '--transfers', '8', '--checkers', '16', *_kg_filters(paths)], quiet=quiet)

def drive_push_bg(*paths):
    # 학습 중 에폭마다: 앞의 올리기가 끝났을 때만 백그라운드로 올림 (학습을 멈추지 않음)
    import threading
    t = _KG.get('bg')
    if not ON_KAGGLE or (t and t.is_alive()):
        return
    _KG['bg'] = threading.Thread(target=drive_push, args=paths, kwargs={'quiet': True}, daemon=True)
    _KG['bg'].start()

def _kg_files(prefix):
    # Drive의 폴더 목록 (경로, 크기, 수정 시각) — 세션마다 한 번만 읽음
    if prefix not in _KG['ls']:
        r = _kg_rc(['lsf', '-R', '--files-only', '--format', 'pst', '--separator', '\t', _kg_remote(prefix)], quiet=True)
        _KG['ls'][prefix] = [tuple(l.split('\t')) for l in r.stdout.splitlines() if l.count('\t') == 2]
    return _KG['ls'][prefix]

_KG_VEXT = ('.mp4', '.mov', '.avi', '.mkv')

def drive_get_videos(query='', files=()):
    # Drive videos/에서 영상을 받음: query = TASK 이름 검색 규칙(파일 이름에 적용) · files = videos/ 기준 경로
    if not ON_KAGGLE:
        return []
    allv = [(p, s) for p, s, t in _kg_files('videos') if p.lower().endswith(_KG_VEXT)]
    pick = [p for p, s in allv if p in set(files)]
    if query:
        pick += [p for p, s in allv if ctt.match_name(os.path.splitext(os.path.basename(p))[0], query)[0]]
    pick = sorted(set(pick))
    if pick:
        mb = sum(int(s) for p, s in allv if p in pick) / 2**20
        print(f'⬇️ 영상 {len(pick)}개 받기 ({mb:.0f} MB) …')
        drive_pull(*[f'videos/{p}' for p in pick])
    return pick

def kg_file_index(*prefixes):
    # Kaggle: Drive 파일 목록 {BASE 기준 경로: (크기, 수정 시각 epoch)} — 새로 읽음 (모델 정리·웹 상태, v4.24.8). Colab은 None
    if not ON_KAGGLE:
        return None
    import datetime as _dt
    out = {}
    for pre in prefixes:
        _KG['ls'].pop(pre, None)
        for p, s, t in _kg_files(pre):
            try:
                ts = _dt.datetime.strptime(t[:19], '%Y-%m-%d %H:%M:%S').timestamp()
            except ValueError:
                ts = 0
            out[f'{pre}/{p}'] = (int(s), ts)
    return out

def kaggle_media_on_drive(source='', tname=''):
    # TASK 원본 영상이 Drive에 있는지만 봄 (받지 않음, v4.24.7): 'videos/<경로>' · 'backups/<경로>' · ''
    if not ON_KAGGLE:
        return ''
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [p for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        return f'videos/{vids[0]}'
    bk = sorted((t, p) for p, s, t in _kg_files('backups') if base and p.startswith(base + '/') and p.endswith('.zip'))
    return f'backups/{bk[-1][1]}' if bk else ''

def kaggle_fetch_media(source='', tname=''):
    # TASK 하나의 원본 영상을 Drive에서 받음: videos/(같은 이름) → 없으면 backups/<기본 이름>/의 가장 최근 백업
    key = (source or '', tname or '')
    if not ON_KAGGLE or key in _KG['media'] or _KG.get('no_fetch'):
        return
    _KG['media'].add(key)
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [(p, s) for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        todo = [p for p, s in vids if not os.path.exists(f'{BASE}/videos/{p}')]
        if todo:
            print(f"⬇️ 영상 받기: {', '.join(todo)} ({sum(int(s) for p, s in vids if p in todo) / 2**20:.0f} MB)")
            drive_pull(*[f'videos/{p}' for p in todo])
        return
    if base:
        bk = sorted((t, p, s) for p, s, t in _kg_files('backups') if p.startswith(base + '/') and p.endswith('.zip'))
        if bk and not os.path.exists(f'{BASE}/backups/{bk[-1][1]}'):
            print(f'⬇️ 백업 받기 (영상 포함): backups/{bk[-1][1]} ({int(bk[-1][2]) / 2**20:.0f} MB)')
            drive_pull(f'backups/{bk[-1][1]}')

def _kg_lock_mirror(run_dir, force=False):
    # Drive의 RUNNING.json을 그대로 비춤 (없으면 로컬도 지움) — 학습 잠금이 다른 컴퓨터와 통하도록
    rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
    if not force and time.time() - _KG['lock_t'].get(rel, 0) < 15:
        return rel
    _KG['lock_t'][rel] = time.time()
    r = _kg_rc(['cat', _kg_remote(rel)], quiet=True)
    p = f'{BASE}/{rel}'
    if r.returncode == 0 and r.stdout.strip():
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, 'w', encoding='utf-8').write(r.stdout)
    elif os.path.exists(p):
        os.remove(p)
    return rel

def _kg_lock_publish(rel):
    p = f'{BASE}/{rel}'
    if os.path.exists(p):
        _kg_rc(['copyto', p, _kg_remote(rel)], quiet=True)
    else:
        _kg_rc(['deletefile', _kg_remote(rel)], quiet=True)
    _KG['lock_t'][rel] = time.time()

def _kg_patch_ctt():
    # ctt(공용 도구)의 잠금·영상 찾기 함수에 Drive 동기화를 덧붙임 (0번 셀이 ctt를 다시 불러올 때마다)
    if not ON_KAGGLE or ctt is None:
        return
    if hasattr(ctt, 'find_video'):
        _fv = ctt.find_video
        def find_video(*a, **k):
            try:
                kaggle_fetch_media(k.get('source', a[1] if len(a) > 1 else ''),
                                   k.get('task_name', k.get('tname', a[2] if len(a) > 2 else '')))
            except Exception as e:
                print(f'(Drive에서 영상 받기 실패: {e})')
            return _fv(*a, **k)
        ctt.find_video = find_video
    if hasattr(ctt, 'video_from_backups'):
        _vb = ctt.video_from_backups
        def video_from_backups(*a, **k):
            try:
                kaggle_fetch_media('', k.get('task_name', a[1] if len(a) > 1 else ''))
            except Exception as e:
                print(f'(Drive에서 백업 받기 실패: {e})')
            return _vb(*a, **k)
        ctt.video_from_backups = video_from_backups
    if all(hasattr(ctt, f) for f in ('acquire_run_lock', 'touch_run_lock', 'release_run_lock', 'read_run_lock')):
        _acq, _touch, _rel, _read = ctt.acquire_run_lock, ctt.touch_run_lock, ctt.release_run_lock, ctt.read_run_lock
        def read_run_lock(run_dir, *a, **k):
            _kg_lock_mirror(run_dir)
            return _read(run_dir, *a, **k)
        def acquire_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            drive_pull(os.path.relpath(run_dir, BASE), quiet=True)     # 이어서 학습할 last.pt·results.csv도 받음
            res = _acq(run_dir, *a, **k)
            if res and res[0]:
                _kg_lock_publish(rel)
            return res
        def touch_run_lock(run_dir, *a, **k):
            rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
            due = time.time() - _KG['lock_t'].get(rel, 0) >= 60          # Drive와 주고받기는 1분에 한 번
            if due:
                _kg_lock_mirror(run_dir, force=True)
            res = _touch(run_dir, *a, **k)
            if due and res:
                _kg_lock_publish(rel)
            return res
        def release_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            res = _rel(run_dir, *a, **k)
            _kg_lock_publish(rel)
            return res
        ctt.read_run_lock, ctt.acquire_run_lock = read_run_lock, acquire_run_lock
        ctt.touch_run_lock, ctt.release_run_lock = touch_run_lock, release_run_lock

def _kg_snapshot():
    snap = {}
    for root, dirs, files in os.walk(BASE):
        rel_root = os.path.relpath(root, BASE)
        if rel_root != '.' and (rel_root + '/').startswith(_KG_NOSYNC):
            dirs[:] = []
            continue
        for f in files:
            p = os.path.join(root, f)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            snap[os.path.relpath(p, BASE)] = (st.st_mtime_ns, st.st_size)
    return snap

def _kg_pre_cell(*_a):
    if not (ON_KAGGLE and KAGGLE_AUTO_SYNC):
        return
    drive_pull(*_KG_SHARED, quiet=True)
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()

def drive_sync_now(echo=True):
    # 지난 스냅숏 뒤에 바뀐 파일을 Drive에 올리고, 셀에서 지운 파일은 Drive에서도 지움(휴지통). 셀이 끝날 때 자동 실행
    snap0 = _KG.pop('snap', None)
    if not ON_KAGGLE or snap0 is None:
        return
    import tempfile
    snap1 = _kg_snapshot()
    _KG.pop('snap_t', 0)
    pulled = _KG.pop('pulled', {})               # 셀 안에서 Drive에서 받은 뒤 그대로인 파일은 빼고
    mine = lambda p: not p.endswith(('.partial', 'RUNNING.json'))   # 학습 잠금은 잠금 함수가 따로 주고받음
    # 셀이 새로 만들거나 바꾼 파일 전부 — 수정 시각을 보존한 복사(shutil.copy2: 14번 셀 export 재사용)도 올림 (v4.24.2, 예전엔 빠졌음)
    changed = sorted(p for p, v in snap1.items() if snap0.get(p) != v and pulled.get(p) != v and mine(p))
    deleted = sorted(p for p in snap0 if p not in snap1 and not p.startswith(_KG_NODELETE) and mine(p))
    for lst, args in ((changed, ['copy', BASE, _kg_remote(), '--update', *(['--no-traverse'] if len(changed) <= 200 else []), '--transfers', '8']),
                      (deleted, ['delete', _kg_remote()])):
        if lst:
            with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False, encoding='utf-8') as fh:
                fh.write('\n'.join(lst) + '\n')
            _kg_rc(args + ['--files-from-raw', fh.name])
            os.remove(fh.name)
    if echo and (changed or deleted):
        mb = sum(snap1[p][1] for p in changed) / 2**20
        print('☁️ Drive 반영: ' + ' · '.join(x for x in (f'올림 {len(changed)}개 ({mb:.1f} MB)' if changed else '',
                                                    f'지움 {len(deleted)}개 (Drive 휴지통)' if deleted else '') if x))

def _kg_post_cell(*_a):
    if ON_KAGGLE and KAGGLE_AUTO_SYNC:
        try:
            drive_sync_now()
        except Exception as e:
            print(f'⚠️ Drive 올리기 실패: {e} → drive_sync_now() 또는 drive_push(경로)로 다시')

def _kg_secret(name):
    # 읽은 값만 기억 — 없던 비밀을 나중에 붙이면 0번 셀을 다시 실행하는 것만으로 읽힘 (v4.13)
    if not _KG['secrets'].get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            _KG['secrets'][name] = UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return _KG['secrets'][name]

def _kg_fix_conf(conf):
    # Kaggle 보안 비밀에 붙여 넣으면 줄바꿈이 사라져 한 줄이 됨 → 항목마다 줄을 나눠 되살림 (v4.12.1)
    conf = conf.replace('\r\n', '\n').strip()
    if len(conf.splitlines()) <= 2:
        conf = re.sub(r'\]\s*', ']\n', conf, count=1)
        conf = re.sub(r'\s+(?=\[[^\]\s]+\]\s)', '\n\n', conf)            # 원격이 여러 개일 때
        conf = re.sub(r'\s+(?=(type|client_id|client_secret|scope|token|team_drive|root_folder_id|service_account_file'
                      r'|service_account_credentials|shared_with_me|remote)\s*=)', '\n', conf)
    return conf + '\n'

def _kg_install_rclone():
    if shutil.which('rclone'):
        return
    import urllib.request, io
    urls = ['https://downloads.rclone.org/rclone-current-linux-amd64.zip']
    try:
        tag = json.load(urllib.request.urlopen('https://api.github.com/repos/rclone/rclone/releases/latest', timeout=30))['tag_name']
        urls.append(f'https://github.com/rclone/rclone/releases/download/{tag}/rclone-{tag}-linux-amd64.zip')
    except Exception:
        pass
    for url in urls:
        try:
            data = urllib.request.urlopen(url, timeout=180).read()
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                exe = z.read([n for n in z.namelist() if n.endswith('/rclone')][0])
            dst = '/usr/local/bin/rclone' if os.access('/usr/local/bin', os.W_OK) else os.path.expanduser('~/.local/bin/rclone')
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            open(dst, 'wb').write(exe)
            os.chmod(dst, 0o755)
            os.environ['PATH'] = os.path.dirname(dst) + ':' + os.environ['PATH']
            print('rclone 설치:', url.split('/')[2])
            return
        except Exception as e:
            print(f'(rclone 받기 실패 {url.split("/")[2]}: {e})')
    raise SystemExit('⛔ rclone을 설치하지 못했습니다 → Kaggle 오른쪽 Session options에서 Internet이 On인지 확인하세요')

if ON_KAGGLE:
    BASE = KAGGLE_BASE
    if DS_ROOT.startswith('/content'):
        DS_ROOT = f'{LOCAL_TMP}/ds'
    os.makedirs(BASE, exist_ok=True)
    _kg_install_rclone()
    _conf = os.environ.get('RCLONE_CONF') or _kg_secret('RCLONE_CONF')
    if not _conf:
        raise SystemExit('⛔ Kaggle 보안 비밀 RCLONE_CONF가 없습니다 → Add-ons → Secrets에 rclone.conf 내용을 넣고, 이 노트북에 체크해 붙이세요 (KAGGLE.md 2장)')
    _conf = _kg_fix_conf(_conf)
    _cp = Path.home() / '.config/rclone/rclone.conf'
    _cp.parent.mkdir(parents=True, exist_ok=True)
    if not _cp.exists() or _cp.read_text() != _conf:
        _cp.write_text(_conf)
    os.chmod(_cp, 0o600)
    _t0 = time.time()
    _r = _kg_rc(['lsf', '--max-depth', '1', _kg_remote('scripts')], quiet=True)
    if _r.returncode != 0:
        _names = re.findall(r'^\[(.+?)\]', _conf, re.M)
        raise SystemExit(f'⛔ Drive {_kg_remote("scripts")} 를 읽지 못했습니다.\n'
                         f'   rclone 오류: {_r.stderr.strip()[-700:]}\n'
                         f'   설정의 원격 이름 {_names} · KAGGLE_REMOTE {KAGGLE_REMOTE!r}\n'
                         '   → notFound: root_folder_id 확인 · accessNotConfigured: GCP에서 Drive API 사용 · '
                         'invalid_grant: 토큰 만료(OAuth 앱을 프로덕션으로, PC에서 rclone config reconnect gdrive:) (KAGGLE.md 2장)')
    drive_pull('scripts', 'config', 'runs/*/*', 'runs/*/weights/best.pt', 'datasets/*.sig', 'web/label_stats.json', *_KG_SHARED, *KAGGLE_PULL)
    if KAGGLE_PULL_DATASET:
        drive_pull('dataset.zip')
    if KAGGLE_VIDEOS:
        drive_get_videos(KAGGLE_VIDEOS)
    print(f'Kaggle: Drive에서 받기 {time.time() - _t0:.0f}초 → {BASE}')
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()   # 0번 셀이 바꾸는 파일(logs/versions.csv 등)도 끝에 올림
    try:
        _ip = get_ipython()
        for _ev, _fn in (('pre_run_cell', _kg_pre_cell), ('post_run_cell', _kg_post_cell)):
            for _old in [f for f in _ip.events.callbacks[_ev] if getattr(f, '__name__', '') == _fn.__name__]:
                _ip.events.unregister(_ev, _old)
            _ip.events.register(_ev, _fn)
    except NameError:
        pass


# Drive 연결
if not ON_KAGGLE and not os.path.exists('/content/drive/MyDrive') and BASE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
def _find_base(base):
    # BASE가 없으면 공유 바로가기·공유 드라이브에서 같은 이름의 폴더를 찾음
    name = os.path.basename(base.rstrip('/'))
    cands = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=False) + \
            glob.glob(f'/content/drive/.shortcut-targets-by-id/*/{name}') + \
            glob.glob(f'/content/drive/Shareddrives/*/{name}')
    return [c for c in cands if os.path.isdir(f'{c}/scripts')]
if not os.path.isdir(BASE):
    found = _find_base(BASE)
    if found:
        print(f'BASE {BASE} 가 없어 찾은 폴더를 씁니다: {found[0]}')
        BASE = found[0]
    elif CREATE_BASE:
        print(f'새 BASE 폴더를 만듭니다: {BASE}')
    else:
        raise SystemExit(
            f'⛔ {BASE} 폴더가 없습니다.\n'
            '   공유받은 폴더라면: Google Drive → 공유 문서함 → eggplant_yolo 오른쪽 클릭 → 정리 → 바로가기 추가 → 내 드라이브\n'
            '   그다음 메뉴 런타임 → 세션 다시 시작 → 0번 셀 다시. (처음 설치하는 사람이면 CREATE_BASE = True)')
if os.path.isdir(BASE) and not glob.glob(f'{BASE}/scripts/*.py'):
    found = [c for c in _find_base(BASE) if glob.glob(f'{c}/scripts/*.py') and os.path.realpath(c) != os.path.realpath(BASE)]
    msg = (f'⚠️ {BASE}/scripts 에 .py가 없습니다. 공유받은 폴더가 아니라 새로 생긴 빈 폴더일 수 있습니다.\n'
           '   빈 eggplant_yolo 폴더를 지우거나 이름을 바꾼 뒤, 공유받은 폴더의 바로가기를 내 드라이브에 추가하세요.')
    if found:
        msg += f'\n   코드가 있는 같은 이름의 폴더: {found[0]} → BASE를 이 경로로 바꿔도 됩니다.'
    print(msg)
for d in ['scripts', 'config', 'videos', 'runs', 'tasks', 'backups', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# 패키지: 없을 때만 설치 (세션마다 확인)
from importlib import metadata
def _ver(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

CVAT_HOST = CVAT_PROFILES[CVAT_AUTH if CVAT_AUTH in CVAT_PROFILES else next(iter(CVAT_PROFILES))]['host']   # 처음 접속할 서버 (셀에서 바뀜)

def cvat_server_version(host=None):
    # CVAT 서버 버전 (/api/server/about, 로그인 불필요). 실패하면 None
    try:
        import requests
        r = requests.get(f'{host or CVAT_HOST}/api/server/about', timeout=15)
        return r.json().get('version') if r.ok else None
    except Exception:
        return None

CVAT_LAST_KNOWN = '2.79.0'                   # 서버 버전을 못 읽을 때 쓸, 마지막으로 시험한 버전
SERVER_VER = cvat_server_version()
if CVAT_SDK_VERSION == 'auto':
    _base = SERVER_VER or CVAT_LAST_KNOWN
    _mj, _mn = _base.split('.')[:2]
    cvat_spec, cvat_prefix = f'cvat-cli~={_mj}.{_mn}.0', f'{_mj}.{_mn}.'   # 같은 X.Y의 최신 패치
else:
    cvat_spec, cvat_prefix = f'cvat-cli=={CVAT_SDK_VERSION}', CVAT_SDK_VERSION
_cli, _sdk = _ver('cvat-cli'), _ver('cvat-sdk')
need = [p for p, mod in [('ultralytics', 'ultralytics'), ('lap', 'lap'), ('av', 'av')] if importlib.util.find_spec(mod) is None]
if not (_cli and _sdk and _cli == _sdk and _cli.startswith(cvat_prefix)):
    need.append(cvat_spec)
if need and not os.environ.get('SKIP_INSTALL'):
    print('설치:', need)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need])
    if r.returncode != 0 and cvat_spec in need:
        # SaaS가 PyPI보다 먼저 올라간 경우 등: 같은 X.Y가 아직 없으면 최신 cvat-cli
        print(f'⚠️ {cvat_spec} 설치 실패 → 최신 cvat-cli 설치')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'cvat-cli',
                        *[p for p in need if p != cvat_spec]], check=True)
    importlib.invalidate_caches()
    if 'cvat_sdk' in sys.modules and sys.modules['cvat_sdk'].__version__ != _ver('cvat-sdk'):
        print('⚠️ 이 세션에 이전 cvat-sdk가 이미 불러져 있습니다 → 메뉴 런타임 → 세션 다시 시작 후 0번 셀을 다시 실행하세요')
elif need:
    print('설치 필요(SKIP_INSTALL이라 건너뜀):', need)
import torch, ultralytics
CVAT_CLI_VER, CVAT_SDK_VER = _ver('cvat-cli'), _ver('cvat-sdk')

def cvat_compat(server_ver=None):
    # cvat-sdk는 서버가 SDK와 같은 X.Y 또는 한 단계 위(X.Y+1)일 때 공식 지원
    server_ver = server_ver or SERVER_VER
    if not (server_ver and CVAT_SDK_VER):
        return '확인 불가'
    s, k = [int(x) for x in server_ver.split('.')[:2]], [int(x) for x in CVAT_SDK_VER.split('.')[:2]]
    if s == k:
        return '일치'
    if s[0] == k[0] and s[1] == k[1] + 1:
        return '지원 범위 (서버가 한 단계 위)'
    return '⚠️ 지원 범위 밖 — CVAT_SDK_VERSION을 서버 버전에 맞추거나 auto로'

# 버전이 바뀔 때만 logs/versions.csv에 한 줄 (문제가 생겼을 때 언제부터인지 추적)
_vrow = [SERVER_VER or '?', CVAT_SDK_VER or '?', CVAT_CLI_VER or '?', ultralytics.__version__]
_vlog = f'{BASE}/logs/versions.csv'
os.makedirs(f'{BASE}/logs', exist_ok=True)
_last = open(_vlog, encoding='utf-8').read().strip().splitlines()[-1].split(',')[1:] if os.path.exists(_vlog) else None
if _last != _vrow:
    import datetime
    with open(_vlog, 'a', encoding='utf-8') as fh:
        if _last is None:
            fh.write('time,cvat_server,cvat_sdk,cvat_cli,ultralytics\n')
        from zoneinfo import ZoneInfo
        fh.write(','.join([datetime.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d %H:%M')] + _vrow) + '\n')

# 데이터셋: dataset.zip이 바뀌었을 때만 다시 풂
DS, yaml_path = None, None
zpath = f'{BASE}/dataset.zip'
stamp = f'{DS_ROOT}/.source_mtime'
if os.path.exists(zpath):
    mt = str(os.path.getmtime(zpath))
    if not (os.path.exists(stamp) and open(stamp).read() == mt):
        shutil.rmtree(DS_ROOT, ignore_errors=True)
        zipfile.ZipFile(zpath).extractall(DS_ROOT)
        open(stamp, 'w').write(mt)
        print('데이터셋 풀기 완료')
    yaml_path = glob.glob(f'{DS_ROOT}/**/data.yaml', recursive=True)[0]
    DS = os.path.dirname(yaml_path)
    txt = open(yaml_path, encoding='utf-8').read()
    txt = re.sub(r'^path:.*$', f'path: {DS}', txt, flags=re.M)     # PC 경로 → Colab 경로
    open(yaml_path, 'w', encoding='utf-8').write(txt)

# 쓸 모델 정하기 (MODEL_CHOICE)
_trained = sorted(glob.glob(f'{BASE}/runs/*/TRAINING_DONE'), key=os.path.getmtime)
if MODEL_CHOICE == 'latest':
    NAME = os.path.basename(os.path.dirname(_trained[-1])) if _trained else MODEL_NAMES.get(1, 'fruit_1920')
elif isinstance(MODEL_CHOICE, int):
    assert MODEL_CHOICE in MODEL_NAMES, f'MODEL_CHOICE {MODEL_CHOICE}가 MODEL_NAMES 번호 {list(MODEL_NAMES)}에 없습니다'
    NAME = MODEL_NAMES[MODEL_CHOICE]
else:
    NAME = str(MODEL_CHOICE)

# 그 모델을 15번 셀로 학습했다면, 그때 만든 데이터셋(datasets/<NAME>.zip)을 DS로 → 2·3번 셀이 같은 데이터로 확인
if ON_KAGGLE and KAGGLE_PULL_DATASET:
    drive_pull(f'datasets/{NAME}.zip')           # Kaggle: 선택 모델의 데이터셋
_ds_zip = f'{BASE}/datasets/{NAME}.zip'
DS_SOURCE = 'dataset.zip' if DS else '없음'
if os.path.exists(_ds_zip):
    _ds_dir = f'{DS_ROOT}_exp/{NAME}'
    if not os.path.exists(f'{_ds_dir}/data.yaml'):
        shutil.rmtree(_ds_dir, ignore_errors=True)
        zipfile.ZipFile(_ds_zip).extractall(_ds_dir)
    yaml_path, DS = f'{_ds_dir}/data.yaml', _ds_dir
    _t = open(yaml_path, encoding='utf-8').read()
    open(yaml_path, 'w', encoding='utf-8').write(re.sub(r'^path:.*$', f'path: {DS}', _t, flags=re.M))
    DS_SOURCE = f'datasets/{NAME}.zip'

# 경로와 상태
RUN_DIR = f'{BASE}/runs/{NAME}'
BEST, LAST, DONE = f'{RUN_DIR}/weights/best.pt', f'{RUN_DIR}/weights/last.pt', f'{RUN_DIR}/TRAINING_DONE'
SCRIPTS = f'{BASE}/scripts'
VIDEOS, TASKS_DIR, REGISTRY = f'{BASE}/videos', f'{BASE}/tasks', f'{BASE}/task_registry.csv'
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
try:
    import cvat_task_tools as ctt
    importlib.reload(ctt)
except ImportError:
    ctt = None
if ctt is not None and hasattr(ctt, 'REVIEW_WORDS'):
    ctt.REVIEWED_RULE = REVIEWED_RULE                # 검색어 '검토' = 이 규칙의 검토 완료 TASK (v4.24)
try:                                            # 웹 관리 화면 연동 (v4.14): 상태 파일·웹 요청
    import web_bridge as wb
    importlib.reload(wb)
except ImportError:
    wb = None
_kg_patch_ctt()                                 # Kaggle: 영상 찾기·학습 잠금이 Drive와 통하게 (Colab에서는 아무것도 안 함)
if ON_KAGGLE and KAGGLE_VIDEOS and ctt is not None:
    drive_get_videos(KAGGLE_VIDEOS)

def epochs_done():
    p = f'{RUN_DIR}/results.csv'
    return max(0, sum(1 for _ in open(p)) - 1) if os.path.exists(p) else 0

def train_status():
    if os.path.exists(DONE):
        return '완료'
    if os.path.exists(LAST):
        return f'중단됨 ({epochs_done()}/{EPOCHS} 에폭) → 1번 셀이 이어서 학습'
    return '시작 전'

def _secret(name):
    # Colab 보안 비밀(왼쪽 열쇠 아이콘) → 없으면 None. 값은 출력하지 않음
    # Kaggle에서는 Kaggle 보안 비밀(Add-ons → Secrets), 둘 다 없으면 환경 변수
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = _kg_secret(name) if ON_KAGGLE else None
        v = v or os.environ.get(name)
    return v.strip().strip("'\"") if v else None

def prof_login(name):
    # 프로필의 로그인 방식 ('token' 또는 'password')
    p = CVAT_PROFILES[name]
    return p.get('login') or ('token' if 'token' in p else 'password')

def _has_secret(name):
    p = CVAT_PROFILES[name]
    return bool(_secret(p['token'])) if prof_login(name) == 'token' else bool(_secret(p['user']) and _secret(p['pass']))

def cvat_mode(mode=None):
    # 쓸 프로필: 셀의 LOGIN → 지금 접속 중인 프로필 → 0번 셀 CVAT_AUTH ('auto' = 보안 비밀이 있는 첫 프로필)
    m = mode or globals().get('CVAT_LOGIN_MODE') or CVAT_AUTH
    if m == 'auto':
        m = next((k for k in CVAT_PROFILES if _has_secret(k)), next(iter(CVAT_PROFILES)))
    assert m in CVAT_PROFILES, f"LOGIN은 'auto' 또는 0번 셀 CVAT_PROFILES의 이름 {list(CVAT_PROFILES)} 중 하나 (지금 값: {m!r})"
    return m

_CRED_TYPED = globals().get('_CRED_TYPED', {})   # 보안 비밀이 없어 입력창으로 받은 값 (이 세션 메모리에만, 0번 셀을 다시 실행해도 유지)

def cvat_auth(mode=None):
    # 프로필의 (로그인 방식, 로그인 정보): ('password', (아이디, 비밀번호)) 또는 ('token', PAT)
    from getpass import getpass
    m = cvat_mode(mode)
    prof, kind = CVAT_PROFILES[m], prof_login(m)
    if kind == 'password':
        u = _secret(prof['user']) or _CRED_TYPED.get((m, 'u')) or input(f"[{m}] CVAT 아이디 ({prof['host']}): ")
        p = _secret(prof['pass']) or _CRED_TYPED.get((m, 'p')) or getpass(f'[{m}] CVAT 비밀번호: ')
        _CRED_TYPED[(m, 'u')], _CRED_TYPED[(m, 'p')] = u, p
        return kind, (u, p)
    t = _secret(prof['token']) or _CRED_TYPED.get((m, 't')) or getpass(f"[{m}] CVAT Personal Access Token ({prof['host']}): ")
    _CRED_TYPED[(m, 't')] = t
    return kind, re.sub(r'^(Token|Bearer)\s+', '', t.strip(), flags=re.IGNORECASE)

def cvat_client(mode=None):
    # cvat-sdk 클라이언트 (그 프로필의 서버·계정으로 로그인)
    m = cvat_mode(mode)
    kind, v = cvat_auth(m)
    host = CVAT_PROFILES[m]['host']
    cl = ctt.connect(host, credentials=v) if kind == 'password' else ctt.connect(host, v)
    apply_scope(cl, m)                          # 기본 조직·프로젝트 (v4.19)
    return cl

_SCOPE_TMP = {}                                  # 웹 요청이 정한 이번 작업만의 조직 (요청이 끝나면 비움)
CVAT_SCOPE = globals().get('CVAT_SCOPE') or {}   # 지금 쓰는 조직·프로젝트 (use_cvat가 정함)

def scope_want(mode=None):
    # 쓸 조직·프로젝트: 웹 요청 → 프로필의 'org'·'project' → 0번 셀 CVAT_ORG·CVAT_PROJECT
    p = CVAT_PROFILES.get(cvat_mode(mode), {})
    if 'org' in _SCOPE_TMP:
        return _SCOPE_TMP['org'], _SCOPE_TMP.get('project', '')
    return p.get('org', globals().get('CVAT_ORG')), p.get('project', globals().get('CVAT_PROJECT', ''))

def apply_scope(cl, mode=None):
    # 클라이언트에 조직·프로젝트 범위를 붙임 (못 찾으면 있는 조직·프로젝트 목록과 함께 멈춤)
    org, proj = scope_want(mode)
    if not hasattr(ctt, 'set_scope'):
        return None
    try:
        return ctt.set_scope(cl, org, proj)
    except ValueError as e:
        raise SystemExit(f'⛔ {e}\n→ 0번 셀 CVAT_ORG·CVAT_PROJECT (또는 프로필의 org·project)를 고치고 0번 셀을 다시 실행하세요')

def cvat_cli_auth(mode=None):
    # cvat-cli에 넘길 인증 인자와 환경 변수 (비밀번호·토큰은 환경 변수로만 전달)
    kind, v = cvat_auth(mode)
    return (['--auth', v[0]], {'PASS': v[1]}) if kind == 'password' else ([], {'CVAT_ACCESS_TOKEN': v})

_CVAT_SESSIONS = globals().get('_CVAT_SESSIONS', {})   # 프로필별 접속 (0번 셀을 다시 실행해도 유지, 로그인 정보가 바뀌면 다시 로그인)

def cvat_forget(name=None):
    # 저장된 접속을 지움 (name=None이면 전부) → 다음 셀에서 다시 로그인
    for k in ([name] if name else list(_CVAT_SESSIONS)):
        _CVAT_SESSIONS.pop(k, None)
    for k in [k for k in _CRED_TYPED if name is None or k[0] == name]:
        _CRED_TYPED.pop(k, None)

def use_cvat(login=None):
    # 이 셀이 쓸 계정·서버를 정함: LOGIN=None이면 지금 그대로, 프로필 이름이면 그 계정으로 바꿈
    global CLIENT, CVAT_LOGIN_MODE, CVAT_HOST, CVAT_SERVER, CVAT_SCOPE
    import hashlib
    m = cvat_mode(login)
    host, kind = CVAT_PROFILES[m]['host'], prof_login(m)
    cred_sig = hashlib.sha256(repr((host,) + cvat_auth(m)).encode()).hexdigest()[:16]   # 값 자체는 저장 안 함
    sess = _CVAT_SESSIONS.get(m)
    if sess is None or sess[3] != cred_sig:
        if sess is not None:
            print(f'[{m}] 로그인 정보가 바뀌어 다시 로그인합니다.')
        try:
            cl = cvat_client(m)
            user = cl.api_client.users_api.retrieve_self()[0].username     # 로그인이 실제로 되는지 확인
        except Exception as e:
            body = str(getattr(e, 'body', '') or '')[:200]
            hint = (f"PAT가 만료됐거나 Read only인지, 보안 비밀 {CVAT_PROFILES[m].get('token')} 이름·값이 맞는지 확인하세요."
                    if kind == 'token' else
                    f"아이디·비밀번호가 틀렸습니다(보안 비밀 {CVAT_PROFILES[m].get('user')}·{CVAT_PROFILES[m].get('pass')} 값, 앞뒤 공백, "
                    f"비밀번호 변경 여부). 입력창으로 넣었다면 cvat_forget('{m}') 후 다시. "
                    'Google 로그인으로 만든 계정은 비밀번호가 없으므로 PAT 프로필로 접속하세요.')
            status = getattr(e, 'status', '')
            raise RuntimeError(f'CVAT 접속 실패 [{m} → {host}] ({type(e).__name__} {status}: {body or str(e)[:200]})\n→ {hint}') from None
        sv = cvat_server_version(host)
        _CVAT_SESSIONS[m] = (cl, f'{user}@{urlparse(host).hostname}', sv, cred_sig)
        if sv and cvat_compat(sv).startswith('⚠️'):
            print(f'⚠️ 이 서버({sv})는 설치된 cvat-sdk {CVAT_SDK_VER}의 지원 범위 밖입니다. 일부 기능이 안 될 수 있습니다.')
    CLIENT, CVAT_SERVER, _sv, _ = _CVAT_SESSIONS[m]
    CVAT_LOGIN_MODE, CVAT_HOST = m, host
    CVAT_SCOPE = apply_scope(CLIENT, m) or {}
    print(f"▶ CVAT 서버: {CVAT_SERVER} · 프로필 {m} ({'PAT' if kind == 'token' else 'ID/PASS'}) · 버전 {_sv or '?'}")
    if hasattr(ctt, 'scope_text'):
        print(f"  범위: {ctt.scope_text(CVAT_SCOPE)}" + (' ← 웹 요청의 조직 (이 작업만)' if 'org' in _SCOPE_TMP else ''))
    return CLIENT

def task_dir(tid):
    # TASK별 작업 폴더. app.cvat.ai가 아닌 서버는 앞에 서버 이름을 붙여 TASK ID가 겹쳐도 섞이지 않게
    host = urlparse(CVAT_HOST).hostname or ''
    tag = '' if host == 'app.cvat.ai' else re.sub(r'[^A-Za-z0-9]+', '-', host) + '_'
    d = f'{TASKS_DIR}/{tag}{tid}'
    if ON_KAGGLE and d not in _KG['tasks']:      # Kaggle: 이 TASK의 작업 폴더를 처음 쓸 때 Drive에서 받음
        _KG['tasks'].add(d)
        drive_pull(os.path.relpath(d, BASE), quiet=True)
    return d

def task_reviewed(name):
    # 검토 완료 TASK인가 (0번 셀 REVIEWED_RULE: 이름의 상태에 그 글자가 있으면)
    return ctt.is_reviewed(name, REVIEWED_RULE) if ctt is not None else True

def task_role(name):
    # 'val' / 'train' / '규칙 밖'(이름 규칙에 안 맞음 → 학습에서는 train) / ''(VAL_RULE 없음)
    return ctt.split_role(name, VAL_RULE) if ctt is not None else ''

def load_script(name):
    # Drive scripts/의 .py를 노트북 안으로 불러옴 (자동 실행 안 됨 → 설정을 바꾼 뒤 함수 호출)
    spec = importlib.util.spec_from_file_location(os.path.splitext(name)[0], f'{SCRIPTS}/{name}')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod

def run(cmd, log=None, env=None):
    # 명령 실행: 화면에 보여 주고, log 경로가 있으면 같은 내용을 파일에도 남김. env = 추가 환경 변수(화면에 안 나옴)
    print('$', ' '.join(map(str, cmd)))
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})   # 진행 출력을 바로바로 보여 줌
    fh = open(log, 'a', encoding='utf-8') if log else None
    if fh:
        fh.write('$ ' + ' '.join(map(str, cmd)) + '\n')
    for line in p.stdout:
        print(line, end='')
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    return p.returncode

def model_label():
    # 대장에 남길 모델 표시: runs/<NAME> + best.pt 수정 시각
    if not os.path.exists(BEST):
        return ''
    import datetime
    return f"{NAME}/best.pt ({datetime.datetime.fromtimestamp(os.path.getmtime(BEST)).strftime('%m-%d %H:%M')})"

# ---------- 학습 잠금 (v4.9): 같은 runs/<이름>에 두 사람이 동시에 쓰지 않게 ----------
import uuid, time
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:8]          # 이 Colab 세션의 표시 (0번 셀을 다시 실행해도 그대로)
if ctt is not None and hasattr(ctt, 'set_shard'):   # v4.31.0: 함께 쓰는 기록은 세션마다 따로 (logs/shards/<이름>/<세션>.csv) → 여러 세션이 동시에 써도 줄이 사라지지 않음
    ctt.set_shard(SESSION_ID)
TAKEOVER_LOCK = globals().get('TAKEOVER_LOCK', False)

def who_am_i():
    return OPERATOR or f'이름없음-{SESSION_ID}'

def lock_run(run_dir, stage):
    # 잠금을 잡음. 다른 사람이 학습 중이면 멈춤(SystemExit)
    held = (ctt.read_run_lock(run_dir) or {}).get('session') == SESSION_ID   # 이미 이 세션이 잡고 있음
    ok, msg = ctt.acquire_run_lock(run_dir, SESSION_ID, who_am_i(), stage, LOCK_STALE_MIN, TAKEOVER_LOCK)
    if not ok:
        raise SystemExit(msg)
    if msg:
        print('ℹ️', msg)
    if LOCK_CONFIRM_SEC and not held:
        print(f'학습 잠금: {os.path.basename(run_dir)} ({who_am_i()}, 세션 {SESSION_ID}) — {LOCK_CONFIRM_SEC}초 뒤 다시 확인 …')
        time.sleep(LOCK_CONFIRM_SEC)
        cur = ctt.read_run_lock(run_dir)
        if cur and cur.get('session') != SESSION_ID:
            raise SystemExit(f"⛔ 거의 같은 때 {cur.get('who')}(세션 {cur.get('session')})도 이 이름으로 시작했습니다 → 이 세션은 멈춥니다. "
                             '다른 TRAIN_NAME을 쓰세요.')

def unlock_run(run_dir):
    ctt.release_run_lock(run_dir, SESSION_ID)

def _lock_callbacks(model, run_dir):
    # 에폭마다 잠금 갱신. 다른 세션이 잠금을 가져갔으면 즉시 멈춤 (결과가 섞이지 않게)
    def hb(trainer):
        ep = int(getattr(trainer, 'epoch', -1)) + 1
        if not ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=ep):
            raise RuntimeError(f'⛔ 다른 세션이 runs/{os.path.basename(run_dir)} 잠금을 가져갔습니다 → 이 학습을 멈춥니다 (결과 섞임 방지)')
    for ev in ('on_pretrain_routine_end', 'on_train_epoch_start', 'on_fit_epoch_end'):
        model.add_callback(ev, hb)
    if ON_KAGGLE:                                # Kaggle: 에폭마다 runs/<이름>을 Drive에 올림 (세션이 끊겨도 이어서 학습)
        model.add_callback('on_fit_epoch_end', lambda tr: drive_push_bg(os.path.relpath(run_dir, BASE)))

def _train_multi_gpu(m, kw, run_dir, n_done, start):
    # v4.30: GPU 여럿으로 동시 학습 (Ultralytics DDP). 학습은 다른 프로세스에서 돌아 에폭 콜백(잠금 갱신·Drive 올리기)이 닿지 않음
    #   → 여기서 1분마다 잠금을 갱신하고 (Kaggle) runs/<이름>을 Drive로. 시작하지 못하면(에폭 0) GPU 하나로 다시 학습
    import threading as _th
    stop = _th.Event()
    def beat():
        while not stop.wait(60):
            try:
                ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=n_done())
                if ON_KAGGLE:
                    drive_push_bg(os.path.relpath(run_dir, BASE))
            except Exception:
                pass
    _th.Thread(target=beat, daemon=True).start()
    print(f"GPU {kw['device'].count(',') + 1}개로 동시 학습 (device={kw['device']}, batch={kw.get('batch')}) — 에폭마다 결과는 runs/{os.path.basename(run_dir)}/results.csv")
    try:
        m.train(**kw)
    except Exception as e:
        if n_done() > 0:
            raise
        print(f"⚠️ GPU 여럿으로 학습을 시작하지 못했습니다 ({type(e).__name__}: {str(e)[:200]}) → GPU 하나로 다시 (batch = 0번 셀 BATCH)")
        kw.update(device=0, batch=BATCH)
        from ultralytics import YOLO
        m2 = YOLO(start or MODEL); _lock_callbacks(m2, run_dir); m2.train(**kw)
    finally:
        stop.set()

def run_training(name, data_yaml, data_sig='', start=None, train_args=None):
    # start = 시작 가중치 (None = 0번 셀 MODEL) · train_args = Ultralytics 학습 설정 덮어쓰기 (lr0·optimizer·epochs 등, v4.24.3)
    # 잠금을 잡고 학습 (완료=건너뜀, 중단=이어서, 처음=새로). 1번·15번 셀이 함께 씀
    from ultralytics import YOLO
    run_dir = f'{BASE}/runs/{name}'
    last, done = f'{run_dir}/weights/last.pt', f'{run_dir}/TRAINING_DONE'
    if os.path.exists(done):
        print('이미 학습 완료 → 건너뜁니다.', open(done, encoding='utf-8').read())
        return False
    lock_run(run_dir, 'train')
    try:
        sp = f'{run_dir}/DATASET_SIG'
        if data_sig:
            old = open(sp).read().strip() if os.path.exists(sp) else ''
            if old and old != data_sig:
                raise SystemExit(f'⛔ runs/{name}은(는) 다른 데이터({old})로 학습하던 이름입니다 (지금 {data_sig}) → TRAIN_NAME을 바꾸세요')
            open(sp, 'w').write(data_sig)
        n_done = lambda: max(0, sum(1 for _ in open(f'{run_dir}/results.csv')) - 1) if os.path.exists(f'{run_dir}/results.csv') else 0
        try:
            if os.path.exists(last):
                print(f'이어서 학습: {n_done()}/{EPOCHS} 에폭부터')
                m = YOLO(last); _lock_callbacks(m, run_dir); m.train(resume=True)
            else:
                m = YOLO(start or MODEL); _lock_callbacks(m, run_dir)
                _kw = dict(data=data_yaml, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                           device=DEVICE, project=f'{BASE}/runs', name=name, exist_ok=True)
                _kw.update(train_args or {})
                if isinstance(_kw.get('device'), str) and ',' in _kw['device']:
                    _train_multi_gpu(m, _kw, run_dir, n_done, start)       # v4.30: GPU 여럿 (Kaggle T4×2)
                else:
                    m.train(**_kw)
        except AssertionError as e:            # 이미 끝난 학습을 resume하려 할 때
            if 'nothing to resume' not in str(e) and 'is finished' not in str(e):
                raise
            print('학습이 이미 끝나 있었습니다.')
        import datetime as _d
        open(done, 'w', encoding='utf-8').write(f'{n_done()} epochs · data {name} · {who_am_i()} · 세션 {SESSION_ID} · '
                                                f'{_d.datetime.now().strftime("%Y-%m-%d %H:%M")}')
        return True
    finally:
        unlock_run(run_dir)

def upload_log(tid, method, model_path=None, echo=print, **ctx):
    # 업로드 한 건의 상세 기록 시작 → .say(메시지), .set(값=…), .finish(상태, 오류)
    #   tasks/<ID>/upload_<시각>.json·.log + logs/uploads.csv (한 줄씩 누적)
    mp = model_path or BEST
    mt, md5 = ctt.file_fingerprint(mp) if mp and os.path.exists(mp) else ('', '')
    return ctt.UploadLog(BASE, task_dir(tid), tid, method, echo=echo, server=CVAT_SERVER, profile=CVAT_LOGIN_MODE,
                         cvat_user=str(CVAT_SERVER).split('@')[0], operator=who_am_i(), session=SESSION_ID,
                         app_version=APP_VERSION, model=os.path.relpath(mp, BASE) if mp else '',
                         model_time=mt, model_md5=md5, **ctx)

def track_settings_of(xml_rel):
    # 5번 셀이 track을 만들 때 남긴 설정 (track_settings.json). 없으면 {}
    p = os.path.join(os.path.dirname(f'{BASE}/{xml_rel}'), 'track_settings.json')
    try:
        return json.load(open(p, encoding='utf-8'))
    except Exception:
        return {}

def change_report(task, before_zip, tag='after', show=True, max_frames=40):
    # 업로드 뒤 라벨을 export해 업로드 전 백업과 비교 → 라벨별·프레임별 '기존→새' 박스 수. CSV는 tasks/<ID>/change_<시각>.csv
    import pandas as pd
    from IPython.display import display
    folder = task_dir(task.id)
    after = ctt.export_annotations(task, f'{folder}/{tag}_{ctt.stamp()}.zip')
    rows, frows, text = ctt.label_change_report(before_zip, after)
    csv_path = ctt.save_change_report(rows, frows, f'{folder}/change_{ctt.stamp()}.csv')
    res = {'text': text, 'rows': rows, 'frames': frows, 'csv': os.path.relpath(csv_path, BASE), 'after': os.path.relpath(after, BASE)}
    if show:
        show_change(task.id, res, max_frames)
    return res

def show_change(tid, res, max_frames=40):
    import pandas as pd
    from IPython.display import display
    print(f"■ TASK {tid} 라벨 변화: {res['text']}")
    display(pd.DataFrame(res['rows']).set_index('라벨'))
    fr = res['frames']
    if fr:
        print(f"프레임별 박스 수 (기존→새){'' if len(fr) <= max_frames else f' — {len(fr)}프레임 중 앞 {max_frames}개, 전체는 CSV'}:")
        display(pd.DataFrame(fr[:max_frames]).set_index('프레임'))
    print(f"   저장: {res['csv']} (라벨별·프레임별 전체) · 업로드 후 라벨 {res['after']}")

_REQ = globals().get('_REQ') or {'next': None}   # 이 세션에서 받은 웹 요청 → 다음 작업 기록(job)에 묶임

def req_use(kind):
    # 웹 요청 받기 (셀의 FROM_REQUESTS = True): 대기 요청을 '진행 중'으로 바꾸고 TASK ID 목록(학습은 설정 dict)을 돌려줌.
    # 그 셀의 작업 기록이 끝날 때(job_end) 요청이 '완료'/'실패'로 바뀜 → 웹 화면 '요청' 탭에 보임
    if wb is None:
        raise SystemExit('⛔ scripts/web_bridge.py가 없습니다 (v4.14 scripts로 바꾸세요)')
    reqs = wb.take(BASE, kind, who_am_i(), SESSION_ID, limit=1 if kind in ('train', 'cleanup') else None,
                   **({'host': 'Kaggle' if ON_KAGGLE else 'Colab'} if 'host' in wb.take.__code__.co_varnames else {}))
    if not reqs:
        raise SystemExit(f"⛔ 받을 웹 요청이 없습니다 ({wb.KINDS[kind][0]}) → 웹 화면에서 요청하거나 FROM_REQUESTS = False")
    _REQ['next'] = (kind, [r['id'] for r in reqs])
    _REQ['params'] = dict(reqs[0].get('params') or {})   # 요청의 설정 (export 형식, 학습 규칙 등)
    for r in reqs:
        print(f"📨 웹 요청 {r['id']} · {wb.KINDS[kind][0]} · {r.get('by', '')} · TASK {len(r.get('task_ids') or [])}개"
              + (f" · {r['note']}" if r.get('note') else ''))
    _SCOPE_TMP.clear()
    if kind != 'train' and _REQ['params'].get('org') is not None:      # 웹에서 고른 TASK들의 조직으로 (프로젝트는 안 따짐) — v4.19
        _SCOPE_TMP.update(org=_REQ['params']['org'], project='')
        print(f"   → 조직: {_REQ['params']['org'] or '개인 작업공간'} (웹 요청의 TASK 조직, 이 작업이 끝나면 0번 셀 기본값으로)")
    if kind in ('train', 'cleanup'):                 # 설정 dict (모델 정리는 v4.24.8)
        return dict(reqs[0].get('params') or {})
    ids = wb.task_ids_of(reqs)
    print(f"   → TASK {len(ids)}개: {ids[:10]}{' …' if len(ids) > 10 else ''}")
    return ids

def _req_bind(job, action):
    nxt = _REQ.get('next')
    if job is not None and nxt and wb is not None and wb.KINDS[nxt[0]][2] == action:
        job['req'], _REQ['next'] = nxt, None

def _req_finish(job, status, summary):
    req = (job or {}).get('req')
    if req:
        _SCOPE_TMP.clear()                       # 웹 요청의 조직 → 다시 0번 셀 기본값
    if req and wb is not None:
        ok = status in ('끝', '완료')
        try:
            done = wb.finish(BASE, req[1], ok=ok, summary=summary or status, job_id=job.get('job_id', ''), who=who_am_i())
            _w = sum(1 for d in done if d.get('status') == 'waiting')
            print(f"📨 웹 요청 {len(done)}건 → " + ('실패' if not ok else '완료' if not _w else
                  f"완료 {len(done) - _w} · 받는 중 {_w} (CVAT가 아직 준비 중 → 다 받으면 완료)") + " (웹 화면 '요청' 탭)")
        except Exception as e:
            print(f'(웹 요청 완료 표시 실패: {e})')

def job_start(cell, action, settings=None, summary=''):
    # 시계열 작업 기록(logs/activity.csv)에 작업 시작 — 이 셀의 TASK별 결과가 이 작업에 묶임 (26번 셀로 보기)
    if ctt is not None and hasattr(ctt, 'start_job'):
        try:
            _j = ctt.start_job(BASE, cell, action, who=who_am_i(), session=SESSION_ID,
                               server=globals().get('CVAT_SERVER', ''), model=model_label() if os.path.exists(BEST) else '',
                               settings=settings, summary=summary)
            _req_bind(_j, action)
            return _j
        except Exception as e:
            print(f'(작업 기록 시작 실패: {e})')

def job_end(status='끝', summary='', **kw):
    if ctt is not None and hasattr(ctt, 'current_job') and ctt.current_job():
        try:
            _j = ctt.current_job()
            ctt.end_job(status, summary, **kw)
            _req_finish(_j, status, summary)
        except Exception as e:
            print(f'(작업 기록 끝 실패: {e})')

def lock_status(name=None):
    # 학습 잠금 상태 한 줄
    d = ctt.read_run_lock(f'{BASE}/runs/{name or NAME}') if ctt else None
    if not d:
        return '없음'
    mine = ' (이 세션)' if d.get('session') == SESSION_ID else ''
    return (f"{d.get('who')}{mine} · 단계 {d.get('stage')} {d.get('epoch', '')} · 마지막 갱신 {d['age_min']:.0f}분 전"
            + ('  ← 오래됨(끊긴 학습)' if d['age_min'] >= LOCK_STALE_MIN else ''))

if ctt is not None and hasattr(ctt, 'import_old_records'):   # 처음 한 번: 예전 기록을 작업 기록(logs/activity.csv)으로
    try:
        ctt.import_old_records(BASE)
    except Exception as _e:
        print(f'(예전 기록 옮기기 실패: {_e})')
if ON_KAGGLE:
    _free = shutil.disk_usage(BASE).free / 2**30
    print(f"실행 환경  : Kaggle ({'Save & Run All' if KAGGLE_BATCH else '대화형'}) · 작업 폴더 {BASE} ↔ Drive {_kg_remote() or KAGGLE_REMOTE} · "
          f"디스크 여유 {_free:.0f} GB · 자동 동기화 {'켬' if KAGGLE_AUTO_SYNC else '끔 → drive_sync_now()'}")
else:
    print('실행 환경  : Colab (Drive 연결)')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ("없음 (오른쪽 Session options → Accelerator → GPU)" if ON_KAGGLE else "없음 (런타임 유형을 T4 GPU로)")}'
      + (f' × {torch.cuda.device_count()} (학습은 1개만 씀)' if torch.cuda.device_count() > 1 else ''))
print(f'패키지     : ultralytics {ultralytics.__version__}, cvat-sdk {CVAT_SDK_VER}, cvat-cli {CVAT_CLI_VER}')
print(f'CVAT 서버  : {SERVER_VER or "버전 확인 불가"} → SDK 호환: {cvat_compat()}')
print(f"CVAT 범위  : 기본 조직 {'구분 안 함 (모든 조직)' if CVAT_ORG is None else (CVAT_ORG or '개인 작업공간')} · "
      f"기본 프로젝트 {CVAT_PROJECT or '전부'}" + (' — 셀이 CVAT에 접속할 때 확인' if CVAT_ORG is not None or CVAT_PROJECT else ''))
if DS:
    print(f'데이터셋   : {DS_SOURCE} → {DS} (train {len(os.listdir(f"{DS}/train/images"))} / val {len(os.listdir(f"{DS}/val/images"))})  ← 2·3번 셀이 쓰는 데이터')
else:
    print('데이터셋   : dataset.zip 없음 (학습을 안 할 거면 괜찮음)')
print(f'선택 모델  : {NAME} (MODEL_CHOICE={MODEL_CHOICE!r})' + ('' if os.path.isdir(RUN_DIR) else '  ⚠️ runs/에 이 폴더가 없습니다 (이름 확인 또는 15번 셀로 학습)'))
print(f'학습 상태  : {train_status()}  [{RUN_DIR}]')
print(f'학습 잠금  : {lock_status()}')
print(f'내 이름    : {who_am_i()} (세션 {SESSION_ID})' + ('' if OPERATOR else "  ← 0번 셀 OPERATOR에 이름을 적으면 잠금 안내에 표시됨"))
print(f'best.pt    : {"있음 — " + model_label() if os.path.exists(BEST) else "없음"}')
if _trained:
    import datetime as _dt
    print('학습된 모델 (최근순, 쓰려면 MODEL_CHOICE에 이름 또는 번호):')
    _num = {v: k for k, v in MODEL_NAMES.items()}
    for d in _trained[::-1][:8]:
        n = os.path.basename(os.path.dirname(d))
        print(f"   {'▶' if n == NAME else ' '} {n:<45} 완료 {_dt.datetime.fromtimestamp(os.path.getmtime(d)).strftime('%m-%d %H:%M')}"
              + (f'  (번호 {_num[n]})' if n in _num else ''))
print(f'videos/    : {sum(1 for f in Path(VIDEOS).rglob("*") if f.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv"))}개 영상 (하위 폴더 포함)')
_m = cvat_mode()
for _k, _p in CVAT_PROFILES.items():
    print(f"CVAT {_k:<8}: {_p['host']} · {'PAT' if prof_login(_k) == 'token' else 'ID/PASS'} · 보안 비밀 "
          f"{'있음' if _has_secret(_k) else '없음(실행 때 입력창)'}" + ('  ← 처음 접속' if _k == _m else ''))
print(f"학습 데이터 : 검토 완료 = 상태에 '{REVIEWED_RULE}' · val = '{VAL_RULE or '무작위(VAL_RATIO)'}' · "
      f"TASK 학습 대장 {len(ctt.read_train_tasks(BASE)) if ctt is not None else 0}줄"
      + (f" · 기록 세션 파일 {ctt.SHARD['id']}" if ctt is not None and hasattr(ctt, 'SHARD') else ''))
if wb is not None:
    _open = [r for r in wb.list_requests(BASE) if r['status'] in ('pending', 'taken')]
    print(f"웹 요청    : 대기·진행 중 {len(_open)}건" + (' → 28번 셀에서 보기, 해당 셀 FROM_REQUESTS = True로 실행' if _open else ''))
print(f'작업 대장  : {REGISTRY if os.path.exists(REGISTRY) else "아직 없음 (첫 작업 때 생성)"}')
_need = ['cvat_task_tools.py', 'cvat_backup.py', 'cvat_to_yolo.py', 'yolo_track_to_cvat.py',
         'eval_occluded_recall.py', 'compare_tracks.py', 'cvat_yolo_function.py', 'diagnose_pipeline.py', 'drive_revisions.py', 'ensemble_annotate.py', 'web_bridge.py', 'relabel_bridge.py']
missing = [f for f in _need if not os.path.exists(f'{SCRIPTS}/{f}')]
def _file_ver(path):
    m = re.search(r'^__version__\s*=\s*"([^"]+)"', open(path, encoding='utf-8').read(), re.M)
    return m.group(1) if m else '버전 없음(예전 파일)'
_wrong = {f: _file_ver(f'{SCRIPTS}/{f}') for f in _need if f not in missing}
SCRIPTS_OK = (APP_VERSION,)                   # v4.31.0: scripts도 바뀜 (세션별 기록 — cvat_task_tools·web_bridge·cvat_backup) → scripts/를 4.30.1로
_wrong = {f: v for f, v in _wrong.items() if v not in SCRIPTS_OK}
_dups = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*(*).py'))
print(f'앱 버전    : {APP_VERSION}')
print('scripts/   : ' + ('모두 있음, 버전 일치' if not missing and not _wrong else
                         ('없음 → ' + ', '.join(missing) + ' ' if missing else '')
                         + ('⚠️ 버전 다름 → ' + ', '.join(f'{f}({v})' for f, v in _wrong.items())
                            + f'  ← Drive scripts/를 v{APP_VERSION} 파일로 바꾸고 세션 다시 시작' if _wrong else '')))
if _dups:
    print(f'⚠️ scripts/에 이름이 겹친 파일: {", ".join(_dups)} → 지우세요 (Drive가 같은 이름 업로드를 새 파일로 만든 것)')
if ON_KAGGLE:                                   # scripts/ 안에 Colab 전용 코드가 남아 있는지 (Kaggle에서 안 될 수 있음)
    _colab_only = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*.py')
                         if os.path.basename(f) != 'drive_revisions.py'
                         and re.search(r'google\.colab|/content/', open(f, encoding='utf-8').read()))
    if _colab_only:
        print(f"⚠️ Colab 전용 코드가 있는 scripts: {', '.join(_colab_only)} → 이 파일을 쓰는 셀은 Kaggle에서 확인 필요")
    print('Kaggle     : 23번 셀(Drive 버전 복원)은 Colab에서만 · 영상·백업은 쓰는 셀이 Drive에서 자동으로 받음 · 직접: drive_pull(경로), drive_get_videos(검색어)')


In [ ]:
# 15. export 폴더로 학습 — zip 목록 확인 → 데이터셋 만들기 → 학습
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
EXP_DIR     = ''        # '' = exports/ 안 가장 최근 폴더 · 폴더 이름 · 여러 폴더 ['20261025_1640_...', '20261025_1702_...']
EXP_DEDUP   = True      # 여러 폴더에 같은 TASK(같은 TASK ID)가 있으면 가장 나중 폴더의 것만 씀 (같은 이미지가 두 번 학습되지 않게)
EXP_QUERY   = ''        # 폴더 안에서 쓸 TASK만 고르기 (모든 셀 같은 검색 규칙, TASK 이름 기준). '' = 폴더 안 전부. 예: 'G1|G3 date=261019.. vf*'
# 검토 완료만 쓰기·train/val 나누기는 0번 셀 REVIEWED_RULE·VAL_RULE (v4.13) — 미검토 TASK는 자동으로 빠지고, VAL_RULE에 맞는 TASK가 val
TRAIN_NAME  = ''        # 학습 결과 이름 runs/<이름> ('' = exp_<폴더 이름>, 다른 데이터로 이미 쓴 이름이면 뒤에 _<서명>을 붙임)
DATASET_NAME = ''       # 데이터셋 이름 datasets/<이름>.zip ('' = TRAIN_NAME과 같음). 이미 만든 데이터셋을 다른 학습 이름으로 쓸 때 그 이름을 적음 (v4.24.6)
                        #   예: TRAIN_NAME = 'fruit_1920_ft261101', DATASET_NAME = 'fruit_r261101_2class_rotate' — EXP_*·CONVERT는 그 데이터셋을 만들 때와 같게
FROM_REQUESTS = False   # True = 웹 화면의 학습 요청을 받아 실행 — 요청의 export 폴더·TASK·검토/val 규칙·학습 이름 (끝나면 요청이 '완료'로)
EXP_IDS     = []        # 쓸 TASK ID만 (비우면 폴더·검색어로 고른 전부). 웹의 '학습 준비'가 준비된 TASK를 넣어 보냄
LIST_ONLY   = False     # True = zip 목록만 보고 끝
BUILD_ONLY  = False     # True = 데이터셋만 만들고 학습은 안 함
REBUILD     = False     # True = 저장된 데이터셋이 있어도 다시 만듦
FORCE_NEW   = False     # True = 같은 이름의 이전 학습(runs/<이름>)과 데이터셋을 지우고 처음부터 (다른 사람이 학습 중이면 안 됨)
ALLOW_CPU   = False     # GPU가 없으면 학습하지 않고 멈춤 (CPU로 imgsz 1920 학습은 며칠 걸림)
CONVERT = {'MERGE_HEALTH': True, 'KEEP_CLASSES': ['healthy_fruit'], 'OCCLUDED_POLICY': 'keep', 'VAL_RATIO': 0.2}   # cvat_to_yolo 설정
# 학습 설정 (v4.24.3) — 같은 데이터셋으로 학습만 달리할 때: TRAIN_NAME은 새 이름, DATASET_NAME = 처음 만든 데이터셋 이름 (v4.24.6)
TRAIN_FROM  = ''        # 시작 가중치: '' = 0번 셀 MODEL(yolo11s.pt)에서 처음부터 · runs/ 이름(예 'fruit_r261101') = 그 모델의 best.pt에서 이어 학습 · 또는 .pt 경로
TRAIN_EPOCHS = None     # None = 0번 셀 EPOCHS · 이어 학습은 보통 짧게 (30~60)
TRAIN_LR0   = None      # 학습률. None = Ultralytics 자동 · 이어 학습 권장 0.001~0.002 · 처음부터는 보통 그대로
TRAIN_OPTIMIZER = 'auto'  # 'auto' · 'SGD' · 'AdamW' — 'auto'는 학습률을 무시하므로 TRAIN_LR0를 정하면 AdamW로 바꿈
TRAIN_ARGS  = {}        # 그 밖의 Ultralytics 학습 설정, 예: {'warmup_epochs': 1, 'lrf': 0.1, 'freeze': 10}
TRAIN_PATIENCE = None   # v4.30: 조기 종료 — 이 에폭 동안 val이 나아지지 않으면 멈춤. None = 0번 셀 PATIENCE · 이어 학습은 10~20
VAL_IDS     = []        # v4.30: val TASK를 ID로 고정 (추가 학습 계획: 기준 모델의 val 그대로) — 정하면 VAL_RULE 대신, 나머지는 모두 train
TRAIN_GPUS  = 'auto'    # v4.30: 'auto' = GPU가 여럿이면(Kaggle T4×2) 모두 써서 동시 학습(DDP) · 1 = GPU 하나만
TRAIN_GPU_BATCH = 4     # GPU 여럿일 때 GPU 하나당 이미지 수 (자동 배치는 GPU 하나에서만 됨) · 메모리 오류면 2
_PLAN = None            # (LC앱 '추가 학습' 탭의 계획 — 웹 요청이 넣음)

if FROM_REQUESTS:                                  # 웹 요청(학습)의 설정
    _p = req_use('train')
    EXP_DIR = _p.get('exp_dirs') or _p.get('exp_dir') or EXP_DIR    # 여러 폴더면 시간 순 (같은 TASK는 나중 폴더 것)
    EXP_IDS = _p.get('task_ids') or EXP_IDS
    if _p.get('reviewed_rule') is not None:
        REVIEWED_RULE = _p['reviewed_rule']
        ctt.REVIEWED_RULE = REVIEWED_RULE
    if _p.get('val_rule') is not None:
        VAL_RULE = _p['val_rule']
    print(f"   → 규칙: 검토 = 상태에 '{REVIEWED_RULE}' · val = '{VAL_RULE}' · TASK {len(EXP_IDS) or '폴더 전부'}개 · 폴더 "
          f"{len(EXP_DIR) if isinstance(EXP_DIR, (list, tuple)) else 1}개 (이 학습에만, 0번 셀 값은 그대로 두려면 0번 셀을 다시 실행)")
    EXP_QUERY = _p.get('exp_query', EXP_QUERY) or ''
    TRAIN_NAME = _p.get('train_name') or TRAIN_NAME
    DATASET_NAME = _p.get('dataset_name') or DATASET_NAME
    BUILD_ONLY = bool(_p.get('build_only', BUILD_ONLY))
    TRAIN_FROM = _p.get('train_from') or TRAIN_FROM                  # 웹 요청의 시작 모델·에폭·학습률 (v4.24.3)
    TRAIN_EPOCHS = _p.get('epochs') or TRAIN_EPOCHS
    TRAIN_LR0 = _p.get('lr0') if _p.get('lr0') not in (None, '') else TRAIN_LR0
    TRAIN_OPTIMIZER = _p.get('optimizer') or TRAIN_OPTIMIZER
    TRAIN_PATIENCE = _p.get('patience') or TRAIN_PATIENCE           # v4.30: 추가 학습 계획
    VAL_IDS = [str(x) for x in (_p.get('val_ids') or VAL_IDS)]
    _PLAN = _p.get('plan') or _PLAN
    if VAL_IDS:
        print(f"   → val = TASK ID {len(VAL_IDS)}개로 고정 (VAL_RULE 대신)" + (f" · 추가 학습 계획 {_PLAN.get('name')} (기준 {_PLAN.get('base')})" if isinstance(_PLAN, dict) else ''))
    print(f'   → EXP_DIR={EXP_DIR!r} · EXP_QUERY={EXP_QUERY!r} · TRAIN_NAME={TRAIN_NAME!r} · BUILD_ONLY={BUILD_ONLY}')
# 학습 설정 정리: 시작 가중치 · Ultralytics에 넘길 값
_TA = dict(TRAIN_ARGS or {})
if TRAIN_EPOCHS:
    _TA['epochs'] = int(TRAIN_EPOCHS)
if TRAIN_LR0 is not None:
    _TA['lr0'] = float(TRAIN_LR0)
    _TA['optimizer'] = TRAIN_OPTIMIZER if TRAIN_OPTIMIZER != 'auto' else 'AdamW'
elif TRAIN_OPTIMIZER != 'auto':
    _TA['optimizer'] = TRAIN_OPTIMIZER
if TRAIN_PATIENCE:
    _TA['patience'] = int(TRAIN_PATIENCE)
import torch as _torch
_NGPU = _torch.cuda.device_count() if _torch.cuda.is_available() else 0
if _NGPU > 1 and str(TRAIN_GPUS) not in ('1', '0', 'None') and 'device' not in _TA:   # v4.30: GPU 여럿으로 동시 학습
    _TA['device'] = ','.join(str(i) for i in range(_NGPU))
    if 'batch' not in _TA and (BATCH is None or int(BATCH) < 1):
        _TA['batch'] = int(TRAIN_GPU_BATCH) * _NGPU
if not TRAIN_FROM:
    _START, _START_TXT = MODEL, f'{MODEL} (처음부터)'
else:
    _t = str(TRAIN_FROM)
    _START = _t if _t.endswith('.pt') else f'{BASE}/runs/{_t}/weights/best.pt'
    if not os.path.isabs(_START) and os.path.exists(f'{BASE}/{_START}'):
        _START = f'{BASE}/{_START}'
    _START_TXT = f"{_t if not _t.endswith('.pt') else os.path.relpath(_START, BASE)} best.pt에서 이어 학습"
print(f"▶ 이번 실행: 폴더 {EXP_DIR or '(가장 최근)'} · 검색어 {EXP_QUERY or '(폴더 전부)'}"
      f"{' · TASK ID ' + str(len(EXP_IDS)) + '개' if EXP_IDS else ''} · 이름 {TRAIN_NAME or '(자동)'} · "
      f"{('데이터셋 ' + DATASET_NAME + ' (있는 것 사용) · ') if DATASET_NAME and DATASET_NAME != TRAIN_NAME else ''}"
      f"{'목록만' if LIST_ONLY else '데이터셋만' if BUILD_ONLY else '데이터셋 + 학습'}"
      + ('' if LIST_ONLY or BUILD_ONLY else f" · 시작 {_START_TXT} · 에폭 {_TA.get('epochs', EPOCHS)} · "
         f"학습률 {_TA.get('lr0', '자동')} ({_TA.get('optimizer', 'auto')})" + (f" · 조기 종료 {_TA['patience']}" if 'patience' in _TA else '') + (f" · GPU {_NGPU}개 동시 (batch {_TA.get('batch')})" if ',' in str(_TA.get('device', '')) else '')
         + (f" · 그 밖 {({k: v for k, v in _TA.items() if k not in ('epochs', 'lr0', 'optimizer')})}" if set(_TA) - {'epochs', 'lr0', 'optimizer'} else '')))
if TRAIN_FROM and not (LIST_ONLY or BUILD_ONLY):
    assert os.path.exists(_START), (f"시작 모델이 없습니다: {_START}\n"
                                    f"runs/ 이름을 확인하세요 (학습을 끝낸 모델: {[os.path.basename(os.path.dirname(os.path.dirname(p))) for p in sorted(glob.glob(f'{BASE}/runs/*/weights/best.pt'))][:12]})")
import xml.etree.ElementTree as ET, hashlib, datetime
from collections import Counter
import pandas as pd
from IPython.display import display

# ---------- 1) 폴더 고르기 ----------
kg_sync_exports()                                 # Kaggle: 다른 세션이 받은 export도 (v4.23)
EXP_ROOT = f'{BASE}/exports'
all_dirs = sorted([d for d in glob.glob(f'{EXP_ROOT}/*') if os.path.isdir(d)], key=os.path.getmtime)
assert all_dirs, f'{EXP_ROOT}에 export 폴더가 없습니다 (14번 셀로 먼저 받기)'
if not EXP_DIR:
    sel = [all_dirs[-1]]
else:
    names = [EXP_DIR] if isinstance(EXP_DIR, str) else list(EXP_DIR)
    sel = [d if os.path.isabs(d) else f'{EXP_ROOT}/{d}' for d in names]
    for d in sel:
        assert os.path.isdir(d), f'폴더 없음: {d}\n있는 폴더(최근 5개): {[os.path.basename(x) for x in all_dirs[-5:]]}'
print('export 폴더:', ', '.join(os.path.basename(d) for d in sel))

# ---------- 2) zip 목록과 이미지 출처 ----------
rows, zips = [], []
_KG['no_fetch'] = ON_KAGGLE                    # Kaggle: 목록 단계에서는 영상을 받지 않음 — Drive에 있는지만 보고, 데이터셋을 새로 만들 때 쓸 TASK 것만 받음 (v4.24.7)
for d in sel:
    for z in sorted(glob.glob(f'{d}/*.zip')):
        zips.append(z)
        r = {'폴더': os.path.basename(d), '파일': os.path.basename(z)}
        try:
            with zipfile.ZipFile(z) as zf:
                xn = [n for n in zf.namelist() if n.replace('\\', '/').split('/')[-1] == 'annotations.xml']
                n_img_in_zip = sum(1 for n in zf.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
                root = ET.fromstring(zf.read(xn[0])) if xn else None
        except Exception as e:
            r.update(형식='읽기 실패', 비고=str(e)[:80]); rows.append(r); continue
        if root is None:
            r.update(형식='annotations.xml 없음'); rows.append(r); continue
        t = root.find('meta/task') if root.find('meta/task') is not None else root.find('meta/job')
        tname = (t.findtext('name') if t is not None else '') or ''
        source = (t.findtext('source') if t is not None else '') or ''
        r['TASK ID'] = (t.findtext('id') if t is not None else '') or ''
        r['_updated'] = (t.findtext('updated') if t is not None else '') or ''   # 이 export가 반영한 CVAT 라벨 시각
        r['_zip'] = z
        if root.find('image') is not None:
            fmt, n_fr = 'images', len(root.findall('image'))
            labels = Counter(b.get('label') for b in root.iter('box'))
        else:
            fmt = 'video'
            ff = (t.findtext('frame_filter') or '') if t is not None else ''
            step = int(ff.split('=')[1]) if ff.startswith('step=') else 1
            n_fr = len(range(int(t.findtext('start_frame') or 0), int(t.findtext('stop_frame') or 0) + 1, step)) if t is not None else 0
            labels = Counter(tr.get('label') for tr in root.findall('track'))
        video = ctt.find_video(VIDEOS, source, tname) if ctt else None
        bk = sorted(glob.glob(f"{BASE}/backups/{ctt.base_name(tname)}/**/*.zip", recursive=True)) if (ctt and tname) else []
        _on_drive = '' if (n_img_in_zip or video or bk) else kaggle_media_on_drive(source, tname)
        src = ('zip 안' if n_img_in_zip else f'영상 {os.path.relpath(str(video), VIDEOS)}' if video
               else f'TASK 백업 영상 ({len(bk)}개 중 최신)' if bk
               else f'{_on_drive} (Drive — 필요할 때 받음)' if _on_drive else '❌ 없음')
        r['_source'], r['_media'] = source, (not n_img_in_zip)
        r.update(형식=fmt, TASK=tname, 프레임=n_fr, 박스_또는_track=sum(labels.values()),
                 라벨=', '.join(f'{k} {v}' for k, v in labels.most_common()), 이미지=src,
                 비고='' if (n_img_in_zip or video or bk or _on_drive) else f"13번 셀로 TASK 백업 또는 videos/에 {ctt.base_name(tname) if ctt else tname}.mp4 (또는 {source}) 올리기")
        rows.append(r)
# 검색어(EXP_QUERY) 밖의 TASK는 뺌
if EXP_QUERY:
    _out = []
    for r in rows:
        if r.get('TASK') and not ctt.match_name(r['TASK'], EXP_QUERY)[0]:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + '검색어 밖 → 제외'
            _out.append(r['_zip'])
    zips = [z for z in zips if z not in _out]
    rows_in = [r for r in rows if r.get('_zip') not in _out]
    print(f"검색어 '{EXP_QUERY}' → 폴더 안 zip {len(rows)}개 중 {len(rows_in)}개 사용")
else:
    rows_in = rows
if EXP_IDS:                                        # TASK ID로 고르기 (웹 '학습 준비'의 준비된 TASK)
    _want = {str(x) for x in ctt.as_task_ids(EXP_IDS)}
    _drop = [r['_zip'] for r in rows_in if r.get('_zip') and str(r.get('TASK ID')) not in _want]
    for r in rows_in:
        if r.get('_zip') in _drop:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + 'TASK ID 목록 밖 → 제외'
    zips = [z for z in zips if z not in _drop]
    rows_in = [r for r in rows_in if r.get('_zip') not in _drop]
    _have = {str(r.get('TASK ID')) for r in rows_in}
    print(f"TASK ID {len(_want)}개 중 이 폴더들에 export가 있는 것 {len(_want & _have)}개"
          + (f" · 없는 것 {len(_want - _have)}개 (export부터): {sorted(_want - _have)[:10]}" if _want - _have else ''))
# 검토 여부·역할 (0번 셀 REVIEWED_RULE·VAL_RULE) — 미검토 TASK는 뺌
for r in rows:
    if r.get('TASK'):
        r['검토'] = '✅' if task_reviewed(r['TASK']) else '⏳ 미검토'
        r['역할'] = task_role(r['TASK']) or '무작위'
if VAL_IDS:                                        # v4.30: val TASK를 ID로 고정 (VAL_RULE 대신)
    _VS = {str(x) for x in VAL_IDS}
    for r in rows:
        if r.get('TASK'):
            r['역할'] = 'val' if str(r.get('TASK ID')) in _VS else 'train'
if REVIEWED_RULE:
    _unrev = [r['_zip'] for r in rows_in if r.get('TASK') and r.get('_zip') and not task_reviewed(r['TASK'])]
    for r in rows_in:
        if r.get('_zip') in _unrev:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + '미검토 → 제외'
    zips = [z for z in zips if z not in _unrev]
    rows_in = [r for r in rows_in if r.get('_zip') not in _unrev]
    if _unrev:
        print(f"미검토 TASK {len(_unrev)}개를 뺐습니다 (상태에 '{REVIEWED_RULE}'가 없음 — 0번 셀 REVIEWED_RULE).")
# 같은 TASK가 여러 폴더에 있으면: 폴더 순서(EXP_DIR에 적은 순서, 기본은 시간순)상 가장 나중 것만 사용
latest = {}
for r in rows_in:
    if r.get('TASK ID'):
        latest[r['TASK ID']] = r['_zip']          # TASK ID 기준 (상태가 바뀌어 이름이 달라도 같은 TASK)
dups = []
for r in rows_in:
    k = r.get('TASK ID')
    if EXP_DEDUP and r.get('TASK ID') and latest.get(k) != r['_zip']:
        r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + f"중복 → 제외 (나중 폴더 {os.path.basename(os.path.dirname(latest[k]))} 것 사용)"
        dups.append(r['_zip'])
if dups:
    zips = [z for z in zips if z not in dups]
    print(f'같은 TASK가 여러 폴더에 있어 이전 것 {len(dups)}개를 뺐습니다 (EXP_DEDUP = False면 모두 사용).')
_KG['no_fetch'] = False
lst = pd.DataFrame(rows).drop(columns=['_zip', '_source', '_media', '_updated'], errors='ignore')
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 50)
display(lst)
n_ok = int(((~lst.get('이미지', pd.Series(dtype=str)).fillna('').str.startswith('❌'))
             & (~lst.get('비고', pd.Series(dtype=str)).fillna('').str.contains('중복|검색어 밖'))).sum()) if len(lst) else 0
print(f'zip {len(zips)}개 사용 · 학습에 쓸 수 있는 것 {n_ok}개 (이미지 없는 것은 자동으로 빠짐)')
_use = [r for r in rows_in if r.get('_zip') in zips and not str(r.get('이미지', '')).startswith('❌')]
if (VAL_RULE or VAL_IDS) and _use:             # train/val 비율 미리 보기 (TASK·프레임)
    _s = pd.DataFrame(_use).assign(역할=lambda d: d['역할'].replace({'규칙 밖': 'train (규칙 밖)'}))
    _s = _s.groupby('역할').agg(TASK=('TASK', 'count'), 프레임=('프레임', 'sum'))
    _s['프레임 비율'] = (100 * _s['프레임'] / max(1, _s['프레임'].sum())).round(1).astype(str) + '%'
    print("■ train / val (val = " + (f"TASK ID {len(VAL_IDS)}개 고정" if VAL_IDS else f"'{VAL_RULE}'") + ")")
    display(_s)
    _out = sorted(r['TASK'] for r in _use if r.get('역할') == '규칙 밖')
    if _out:
        print(f'참고: 이름 규칙 밖 TASK {len(_out)}개는 val 규칙을 볼 수 없어 train으로 씁니다: {_out[:5]}')
    if not any(r.get('역할') == 'val' for r in _use):
        print(f"⚠️ val TASK가 없습니다 → 0번 셀 VAL_RULE '{VAL_RULE}'를 확인하세요 (학습하면 멈춤)")

# ---------- 3) 데이터셋 ----------
if not LIST_ONLY and zips:
    job_start(15, 'train', {'exp_dir': [os.path.basename(d) for d in sel], 'query': EXP_QUERY, 'dedup': EXP_DEDUP,
                            'train_name': TRAIN_NAME, 'dataset_name': DATASET_NAME or TRAIN_NAME, 'convert': CONVERT, 'model': MODEL, 'imgsz': IMGSZ, 'epochs': EPOCHS,
                            'batch': BATCH, 'patience': PATIENCE, 'build_only': BUILD_ONLY,
                            'reviewed_rule': REVIEWED_RULE, 'val_rule': VAL_RULE, 'val_ids': len(VAL_IDS),
                            'plan': (_PLAN.get('name') if isinstance(_PLAN, dict) else _PLAN), 'gpus': _TA.get('device', ''),
                            'train_from': (os.path.relpath(_START, BASE) if TRAIN_FROM else MODEL), 'train_args': _TA})
    for r in rows:                                             # 어떤 TASK가 학습 데이터로 쓰였나 (TASK별 기록)
        if not r.get('TASK ID'):
            continue
        _used = r.get('_zip') in zips and not str(r.get('이미지', '')).startswith('❌')
        ctt.log_task('train_data', r['TASK ID'], r.get('TASK', ''), '사용' if _used else '제외', frames=r.get('프레임', ''),
                     boxes=r.get('박스_또는_track', ''), by_label=r.get('라벨', ''),
                     file=os.path.relpath(r['_zip'], BASE) if r.get('_zip') else '', source=r.get('이미지', ''),
                     summary=r.get('비고', ''))
    tag = os.path.basename(sel[-1]) if len(sel) == 1 else f'{len(sel)}dirs_{os.path.basename(sel[-1])}'
    sig = hashlib.md5(json.dumps([[os.path.basename(z), os.path.getsize(z)] for z in zips] + [CONVERT, IMGSZ, EXP_QUERY]
                                 + ([REVIEWED_RULE, VAL_RULE] if (REVIEWED_RULE or VAL_RULE) else [])
                                 + ([sorted(str(x) for x in VAL_IDS)] if VAL_IDS else []),
                                 sort_keys=True).encode()).hexdigest()[:12]
    def _used_sig(n):
        # 그 이름이 이미 쓰인 데이터 서명 (runs/<n>/DATASET_SIG → datasets/<n>.sig). 없으면 ''
        for p in (f'{BASE}/runs/{n}/DATASET_SIG', f'{BASE}/datasets/{n}.sig'):
            if os.path.exists(p):
                return open(p).read().strip()
        return 'unknown' if os.path.exists(f'{BASE}/runs/{n}/weights') else ''
    if not TRAIN_NAME:                                         # 자동 이름: 다른 데이터로 쓴 이름이면 서명을 붙여 새 이름
        TRAIN_NAME = f'exp_{tag}'
        if _used_sig(TRAIN_NAME) not in ('', sig):
            TRAIN_NAME = f'exp_{tag}_{sig[:6]}'
            print(f'같은 폴더 이름의 이전 학습과 데이터가 달라 새 이름을 씁니다: {TRAIN_NAME}')
    _u = _used_sig(TRAIN_NAME)
    if _u not in ('', sig) and not FORCE_NEW:
        raise SystemExit(f"⛔ TRAIN_NAME '{TRAIN_NAME}'은(는) 다른 데이터(서명 {_u})로 이미 쓴 이름입니다 (지금 데이터 서명 {sig}).\n"
                         f"   덮어쓰면 그 결과가 사라집니다 → TRAIN_NAME을 새 이름으로 바꾸세요 (예: '{TRAIN_NAME}_2').\n"
                         '   정말 지우고 다시 하려면 FORCE_NEW = True')
    RUN_LOCK_DIR = f'{BASE}/runs/{TRAIN_NAME}'
    if FORCE_NEW or not os.path.exists(f'{RUN_LOCK_DIR}/TRAINING_DONE'):
        lock_run(RUN_LOCK_DIR, 'dataset')                      # 데이터셋을 만드는 동안에도 다른 사람이 같은 이름을 못 씀
    if FORCE_NEW and _u not in ('', sig):
        for _p in (RUN_LOCK_DIR + '/weights', RUN_LOCK_DIR + '/results.csv', RUN_LOCK_DIR + '/TRAINING_DONE', RUN_LOCK_DIR + '/DATASET_SIG'):
            shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) else (os.remove(_p) if os.path.exists(_p) else None)
        print(f'FORCE_NEW: runs/{TRAIN_NAME}의 이전 학습을 지웠습니다')
    DS_NAME = DATASET_NAME or TRAIN_NAME                       # v4.24.6: 데이터셋 이름 (학습 이름과 따로 둘 수 있음)
    _other_ds = DS_NAME != TRAIN_NAME
    ds_zip, ds_sig = f'{BASE}/datasets/{DS_NAME}.zip', f'{BASE}/datasets/{DS_NAME}.sig'
    ds_json = f'{BASE}/datasets/{DS_NAME}.json'                # 데이터셋을 만든 설정 (v4.24.6~)
    ds_dir = f'{DS_ROOT}_exp/{DS_NAME}'
    os.makedirs(f'{BASE}/datasets', exist_ok=True)
    if _other_ds:
        drive_pull(f'datasets/{DS_NAME}.sig', f'datasets/{DS_NAME}.json', quiet=True)
    if ON_KAGGLE and not REBUILD and os.path.exists(ds_sig) and open(ds_sig).read().strip() == sig:
        drive_pull(f'datasets/{DS_NAME}.zip')   # Kaggle: 같은 데이터로 만든 데이터셋이 Drive에 있으면 받아서 씀
    saved_ok = os.path.exists(ds_zip) and os.path.exists(ds_sig) and open(ds_sig).read().strip() == sig
    if _other_ds and not saved_ok:                             # 남의 이름의 데이터셋은 새로 만들거나 덮어쓰지 않음
        unlock_run(RUN_LOCK_DIR)
        if not os.path.exists(ds_zip):
            raise SystemExit(f'⛔ DATASET_NAME 데이터셋이 없습니다: datasets/{DS_NAME}.zip → 이름을 확인하거나 DATASET_NAME = \'\'로 새로 만드세요')
        _was = json.load(open(ds_json, encoding='utf-8')) if os.path.exists(ds_json) else None
        _now = {'EXP_DIR': [os.path.basename(d) for d in sel], 'EXP_QUERY': EXP_QUERY, 'EXP_DEDUP': EXP_DEDUP, 'EXP_IDS': len(EXP_IDS),
                'CONVERT': CONVERT, 'IMGSZ': IMGSZ, 'REVIEWED_RULE': REVIEWED_RULE, 'VAL_RULE': VAL_RULE, 'zip 수': len(zips)}
        print('데이터셋을 만든 설정과 지금 설정:')
        for _k in _now:
            _w = (_was or {}).get(_k, '(기록 없음)')
            print(f"  {'  ' if _w == _now[_k] or _w == '(기록 없음)' else '≠ '}{_k}: 만들 때 {_w!r} · 지금 {_now[_k]!r}")
        raise SystemExit(f'⛔ datasets/{DS_NAME}.zip은 지금 설정과 다른 데이터로 만든 것입니다 (서명 {open(ds_sig).read().strip() if os.path.exists(ds_sig) else "없음"} ≠ 지금 {sig}).\n'
                         '   EXP_DIR·EXP_QUERY·EXP_DEDUP·EXP_IDS·CONVERT와 0번 셀 IMGSZ·REVIEWED_RULE·VAL_RULE을 만들 때와 같게 하세요'
                         + ('' if _was else ' (이 데이터셋은 v4.24.6 이전에 만들어 설정 기록이 없음 → 그때 15번 셀 설정 그대로)') + '.')
    if _other_ds and REBUILD:
        unlock_run(RUN_LOCK_DIR)
        raise SystemExit(f'⛔ REBUILD = True와 DATASET_NAME을 함께 쓸 수 없습니다 (다른 학습의 데이터셋 datasets/{DS_NAME}.zip을 덮어씀)')
    if saved_ok and not REBUILD:
        if not os.path.exists(f'{ds_dir}/data.yaml'):
            shutil.rmtree(ds_dir, ignore_errors=True)
            zipfile.ZipFile(ds_zip).extractall(ds_dir)
        print(f'저장된 데이터셋 사용: datasets/{DS_NAME}.zip (zip 목록·설정 같음)' + (f' → 학습 이름 {TRAIN_NAME}' if _other_ds else ''))
    else:
        in_dir = f'{LOCAL_TMP}/exp_in/{DS_NAME}'                  # 여러 폴더의 zip을 한곳에 (복사, 라벨이라 작음)
        shutil.rmtree(in_dir, ignore_errors=True); os.makedirs(in_dir)
        for z in zips:
            dst = f'{in_dir}/{os.path.basename(z)}'
            if os.path.exists(dst):
                dst = f'{in_dir}/{os.path.basename(os.path.dirname(z))}__{os.path.basename(z)}'
            shutil.copy2(z, dst)
        shutil.rmtree(ds_dir, ignore_errors=True)
        if ON_KAGGLE:                                  # v4.24.7: 이번 데이터셋에 쓰는 TASK 중 영상에서 프레임을 꺼낼 것만 Drive에서 받음
            _need = [r for r in _use if r.get('_media')]
            if _need:
                print(f'영상에서 프레임을 꺼낼 TASK {len(_need)}개의 영상을 Drive에서 받습니다 (이미 받은 것은 건너뜀)')
            for r in _need:
                kaggle_fetch_media(r.get('_source', ''), r.get('TASK', ''))
        conv = load_script('cvat_to_yolo.py')
        conv.INPUT_DIR, conv.VIDEOS_DIR, conv.OUT_DIR = Path(in_dir), Path(VIDEOS), Path(ds_dir)
        conv.BACKUPS_DIR = Path(f'{BASE}/backups')     # 라벨만 받은 zip: videos/에 없으면 TASK 백업의 원본 영상에서 프레임 추출
        conv.DECODER = 'auto'                          # PyAV(CVAT와 같은 디코딩)로 프레임 번호를 맞춤
        conv.IMAGES_DIR, conv.CVAT_FRAMES_DIR = Path(f'{BASE}/raw_images'), Path(f'{BASE}/cvat_frames')
        conv.TRAIN_IMGSZ = IMGSZ
        _lost = []
        conv.PROGRESS_HOOK = lambda: ctt.touch_run_lock(RUN_LOCK_DIR, SESSION_ID, stage='dataset') or _lost.append(1)
        for k, v in CONVERT.items():
            setattr(conv, k, v)
        if VAL_RULE or VAL_IDS:                        # v4.13: VAL_RULE에 맞는 TASK(의 영상 전체)를 val로 · v4.30: VAL_IDS
            conv.VAL_TASKS = {str(r['TASK ID']): r.get('역할') == 'val' for r in _use if r.get('TASK ID')}
            if not any(conv.VAL_TASKS.values()):
                unlock_run(RUN_LOCK_DIR)
                raise SystemExit(f"⛔ VAL_RULE '{VAL_RULE}'에 맞는 TASK가 없습니다 → 0번 셀 VAL_RULE을 고치거나 '' (무작위)")
        else:
            conv.VAL_TASKS = None
        conv.main()
        if _lost:
            raise SystemExit('⛔ 데이터셋을 만드는 동안 다른 세션이 이 이름의 잠금을 가져갔습니다 → 저장하지 않고 멈춥니다')
        print('데이터셋 저장 중 → Drive datasets/ ...')
        base_noext = ds_zip[:-4]
        shutil.make_archive(base_noext, 'zip', ds_dir)
        open(ds_sig, 'w').write(sig)
        json.dump({'EXP_DIR': [os.path.basename(d) for d in sel], 'EXP_QUERY': EXP_QUERY, 'EXP_DEDUP': EXP_DEDUP, 'EXP_IDS': len(EXP_IDS),
                   'CONVERT': CONVERT, 'IMGSZ': IMGSZ, 'REVIEWED_RULE': REVIEWED_RULE, 'VAL_RULE': VAL_RULE, 'zip 수': len(zips),
                   'VAL_IDS': sorted(str(x) for x in VAL_IDS), 'sig': sig, 'made_by': TRAIN_NAME, 'time': ctt.now(), 'app_version': APP_VERSION},
                  open(ds_json, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)   # v4.24.6: 이 데이터셋을 만든 설정
        if os.path.exists(f'{ds_dir}/stats_report.html'):
            shutil.copy2(f'{ds_dir}/stats_report.html', f'{BASE}/datasets/{DS_NAME}_report.html')
        print(f'저장: datasets/{DS_NAME}.zip · 통계 리포트 datasets/{DS_NAME}_report.html · 설정 datasets/{DS_NAME}.json')
    yaml_path = f'{ds_dir}/data.yaml'
    txt = open(yaml_path, encoding='utf-8').read()
    open(yaml_path, 'w', encoding='utf-8').write(re.sub(r'^path:.*$', f'path: {ds_dir}', txt, flags=re.M))
    DS = ds_dir
    n_tr, n_va = len(os.listdir(f'{DS}/train/images')), len(os.listdir(f'{DS}/val/images'))
    print(f'데이터셋: train {n_tr}장 / val {n_va}장')
    ctt.log_task('dataset', '', TRAIN_NAME, '준비', frames=n_tr + n_va, summary=f'train {n_tr} / val {n_va} · 서명 {sig}'
                 + (f' · 데이터셋 {DS_NAME}' if _other_ds else ''),
                 file=os.path.relpath(ds_zip, BASE))
    _split_csv = f'{ds_dir}/split_tasks.csv'          # v4.13: TASK 학습 대장 (logs/train_tasks.csv)
    if os.path.exists(_split_csv):
        _info = {str(r['TASK ID']): {'state': (ctt.parse_task_name(r.get('TASK', '')) or {'state': '규칙 밖'}).get('state', ''),
                                     'label_updated': r.get('_updated', ''), 'export_zip': os.path.relpath(r['_zip'], BASE)}
                 for r in rows if r.get('TASK ID') and r.get('_zip') in zips}
        _n = ctt.append_train_tasks(BASE, TRAIN_NAME, sig, _split_csv, info=_info, who=who_am_i(), session=SESSION_ID,
                                    app_version=APP_VERSION)
        _sp = pd.read_csv(_split_csv)
        if 'frames_from' in _sp:                    # v4.24.4: 학습 이미지 출처·디코더 (TASK 학습 대장 frames_from 열)
            print('학습 이미지 출처: ' + ', '.join(f'{k} {n}개 TASK' for k, n in _sp['frames_from'].fillna('?').value_counts().items()))
        print(f"TASK 학습 대장: {'새로 ' + str(_n) + '줄' if _n else '이미 기록됨'} · TASK train {int((_sp['split'] == 'train').sum())}개 / "
              f"val {int((_sp['split'] == 'val').sum())}개 (logs/train_tasks.csv, 22번 셀에서 보기)")
    else:
        print('참고: 이 데이터셋은 v4.13 이전에 만들어 TASK별 train/val 기록(split_tasks.csv)이 없습니다 → REBUILD = True로 다시 만들면 기록됨')
    if n_tr == 0 or n_va == 0:
        unlock_run(RUN_LOCK_DIR)
    assert n_tr > 0, '학습 이미지가 0장입니다 — 위 표의 이미지 열(❌ 없음)을 확인하세요'
    assert n_va > 0, 'val 이미지가 0장입니다 — 이미지가 있는 TASK가 너무 적습니다 (VAL_RATIO 또는 zip 확인)'

    # 이후 1·2·3번 셀도 이 학습을 기준으로
    NAME = TRAIN_NAME
    RUN_DIR = f'{BASE}/runs/{NAME}'
    BEST, LAST, DONE = f'{RUN_DIR}/weights/best.pt', f'{RUN_DIR}/weights/last.pt', f'{RUN_DIR}/TRAINING_DONE'
    print(f'학습 이름: {NAME} · 상태: {train_status()}')

    # ---------- 4) 학습 ----------
    if not BUILD_ONLY:
        import torch
        from ultralytics import YOLO
        if not torch.cuda.is_available() and not ALLOW_CPU:
            unlock_run(RUN_LOCK_DIR)
            raise SystemExit('⛔ GPU가 없습니다. 메뉴 런타임 → 런타임 유형 변경 → T4 GPU → 0번 셀 → 15번 셀을 다시 실행하세요.\n'
                             '   (데이터셋은 Drive에 저장돼 있어 다시 만들지 않습니다. 정말 CPU로 하려면 ALLOW_CPU = True)')
        print(f"학습 장치: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (느림)'}")
        if FORCE_NEW and os.path.exists(LAST) and _u in ('', sig):
            for _p in (RUN_DIR + '/weights', RUN_DIR + '/results.csv', RUN_DIR + '/TRAINING_DONE'):
                shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) else (os.remove(_p) if os.path.exists(_p) else None)
            print(f'이전 학습 삭제: runs/{NAME}')
        if os.path.exists(DONE):
            print('이미 학습 완료 → 건너뜁니다. (다시 하려면 TRAIN_NAME을 바꾸거나 FORCE_NEW = True)')
        elif run_training(NAME, yaml_path, data_sig=sig, start=_START, train_args=_TA):
            print('학습 완료:', BEST, '→ 2번 셀(결과), 3번 셀(검증 1단계)')
    unlock_run(RUN_LOCK_DIR)                                   # 데이터셋만 만들었거나 건너뛴 경우에도 잠금을 풂
    _res = f'{BASE}/runs/{TRAIN_NAME}/results.csv'
    _best = ''
    try:
        _df = pd.read_csv(_res); _df.columns = [c.strip() for c in _df.columns]
        _k = 'metrics/mAP50-95(B)'
        if _k in _df:
            _i = _df[_k].idxmax()
            _best = f"에폭 {len(_df)}개 · 최고 mAP50-95 {_df[_k].max():.3f} (에폭 {int(_df.loc[_i, 'epoch'])}) · mAP50 {_df.loc[_i, 'metrics/mAP50(B)']:.3f}"
    except Exception:
        pass
    job_end('끝' if os.path.exists(DONE) or BUILD_ONLY else '중단?', f"runs/{TRAIN_NAME} · " + (_best or ('데이터셋만' if BUILD_ONLY else '결과 없음')),
            file=f'runs/{TRAIN_NAME}')


In [ ]:
# 끝 — 남은 파일을 Drive에 올림 (자동 동기화가 이미 했어도 안전)
drive_sync_now()
drive_push(f'runs/{NAME}', 'datasets/*.zip', 'datasets/*.sig', 'datasets/*_report.html', 'logs', 'task_registry.csv')
print('완료 → Drive runs/' + NAME)
